## Band NN Testing

In [1]:
import process

In [2]:
band = process.process("Adele") # 26 songs, each song w/ ~100 chords

In [3]:
band

[['Am',
  'Em7',
  'Am',
  'Em',
  'Dm',
  'E7',
  'Dm',
  'Em',
  'G7',
  'C',
  'Em',
  'C',
  'Em7',
  'Am',
  'E7',
  'F7',
  'Em',
  'G7',
  'F',
  'Em',
  'Am7',
  'D7sus4',
  'F',
  'Em',
  'E7',
  'F',
  'Em',
  'Am7',
  'D7sus4',
  'Dm',
  'E7',
  'F',
  'Em',
  'Am7',
  'D7sus4',
  'Em',
  'C',
  'Am',
  'Em',
  'Dm',
  'E7',
  'F7',
  'Em',
  'G7',
  'F9',
  'Em7',
  'Dm7',
  'F',
  'E7',
  'F9',
  'Em',
  'Am7',
  'D7sus4',
  'Em7',
  'Am7',
  'D7sus4',
  'F',
  'F7',
  'G7'],
 ['C',
  'G',
  'Bb',
  'F',
  'Fm',
  'C',
  'D7',
  'G7',
  'C',
  'C',
  'G',
  'Bb',
  'F',
  'Fm',
  'C',
  'D7',
  'G7',
  'C',
  'F',
  'C',
  'E7',
  'F',
  'C',
  'F',
  'C',
  'D7',
  'G7',
  'C',
  'G',
  'Bb',
  'F',
  'Fm',
  'C',
  'D7',
  'G7',
  'C',
  'C',
  'G',
  'Bb',
  'F',
  'Fm',
  'C',
  'D7',
  'G7',
  'C',
  'F',
  'C',
  'E7',
  'F',
  'C',
  'F',
  'C',
  'D7',
  'G7',
  'C',
  'G',
  'Bb',
  'F',
  'Fm',
  'C',
  'D7',
  'G7',
  'C',
  'D7',
  'G7',
  'C'],
 ['C',
  'G',
 

In [4]:
# swift = edsheeran

In [8]:
avoid = ["C#", "F#m7b5", "Fb", "Bbm7", "Db", "F#"]

# triple for loop but not much I can do, sigh
for s in range(len(band)):
    for c in range(len(band[s])):
        for ch in avoid:
            if ch in band[s][c]:
                band[s][c] = "C"
    

In [9]:
all = []
for song in band:
    all.extend(song)
all = set(all)

In [10]:
print(all)
print(len(all))

{'Bb7', 'Ab7', 'E5', 'C4', 'A', 'Dm7', 'Em7', 'G', 'E7', 'E', 'Eaug', 'A5', 'G5', 'Ab', 'Em', 'Cm', 'F7', 'Bb', 'Emadd6', 'Gsus4', 'Fm', 'A7', 'Am7', 'Ebm', 'Ab7sus4', 'Bbbm', 'Gbm7', 'Gadd9', 'G7', 'Dmadd9', 'G#m', 'Esus4', 'B', 'G9', 'F', 'Am', 'Dm', 'Csus4', 'C', 'D7sus4', 'D', 'Bsus', 'Gbmadd6', 'F9', 'D7', 'G4', 'Gbm', 'F7b5', 'Asus4', 'C7'}
50


In [11]:
itoc = {x+1:y for x, y in enumerate(all)}
itoc[0] = "."
ctoi = {x:y for y,x in itoc.items()}

In [12]:
# Standard imports 
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import numpy as np
import matplotlib.pyplot
%matplotlib inline

In [13]:
X = []
Y = []

block_size = 3
# build big dataset
for s in band:
    context = [0] * 3
    pred = ""
    for ch in s:
        pred = ctoi[ch]
        X.append(context)
        Y.append(pred)
        context = context[1:] + [ctoi[ch]]

Xtr = torch.tensor(X[:int(0.9 * len(X))])
Ytr = torch.tensor(Y[:int(0.9 * len(Y))])

Xdev = torch.tensor(X[int(0.9 *len(X)):])
Ydev = torch.tensor(Y[int(0.9 *len(Y)):])

In [14]:
# Linear class according to torch.nn
class Linear:
    
    def __init__(self, in_features, out_features, bias=True):
        self.weight = torch.randn(in_features, out_features)
        self.bias = torch.randn(out_features) if bias else None
        
    def __call__(self, x):
        self.out = x @ self.weight
        if self.bias is not None:
            self.out += self.bias
        return self.out

    def parameters(self):
        if self.bias is None:
            return [self.weight]
        return [self.weight] + [self.bias]
    
    
class BatchNorm1d:
    
    def __init__(self, num_features, eps=1e-5, momentum=0.1,):
        self.training = True
        self.eps = eps
        self.momentum = momentum 
        self.gamma = torch.ones(num_features) # multiplying factor on BatchNorm
        self.beta = torch.zeros(num_features) # additive factor on BatchNorm
        
        # during test time 
        self.running_mean = torch.zeros(num_features) 
        self.running_var = torch.ones(num_features)
        
    def __call__(self, x):
        # do different things during training and testing
        if self.training:
            mean = x.mean(0, keepdim=True)
            var = x.var(0, keepdim=True)
        else:
            mean = self.running_mean
            var = self.running_var
            
        self.out = (x - mean)/torch.sqrt(var + self.eps) # normalize to maintain unit Gaussian
        self.out = self.gamma * self.out + self.beta # apply gain and bias
        
        if self.training:
            with torch.no_grad():
                # update running_mean and running_var
                self.running_mean = self.running_mean * (1-self.momentum) + mean*self.momentum
                self.running_var = self.running_var * (1-self.momentum) + var*self.momentum
            
        return self.out
    
    def parameters(self):
        return [self.gamma] + [self.beta]
    
    
class Embedding:
    
    def __init__(self, num_embeddings, embedding_dim):
        self.weight = torch.randn(num_embeddings, embedding_dim)
    
    def __call__(self, IX):
        self.out = self.weight[IX]
        return self.out
    
    def parameters(self):
        return [self.weight]
    

class Flatten:        
    
    def __call__(self, IX):
        self.out = IX.view(IX.shape[0], -1) # make 2d
        return self.out
        
    def parameters(self):
        return []
    
    
class Tanh:
    
    def __call__(self, x):
        self.out = torch.tanh(x) 
        return self.out
    
    def parameters(self):
        return []

In [55]:
# Define parameters
block_size = block_size
vocab_size = len(ctoi)
n_embd = 2
n_hidden = 25
batch_size = 20
num_hidden = 1

In [60]:
layers = [
    Embedding(vocab_size, n_embd), # num_emeddings = vocab_size and embedding_dim = n_embd
    Flatten(),
    Linear(block_size * n_embd, n_hidden), 
    BatchNorm1d(n_hidden),
    Tanh(),
    Linear(n_hidden, vocab_size)
]

with torch.no_grad():
    # fix some things
    layers[-1].weight *= 0.01 # fix last layer confidently wrong 
    if layers[-1].bias is not None:
        layers[-1].bias *= 0
    for layer in layers:
        if isinstance(layer, Linear):
            layer.weight *= 0.2
            if layers[-1].bias is not None:
                layer.bias *= 0.2
    
    
# parameters and stuff
parameters = [p for layer in layers for p in layer.parameters()]
for p in parameters:
    p.requires_grad = True

In [73]:
# Training loop

num_iters = 200
for i in range(num_iters):
    
    # generate batches
    nums = torch.randint(0, Xtr.shape[0], (batch_size,))
    x = Xtr[nums] # batch we will work with 
    y = Ytr[nums]
    
    # apply feed-forward process
    for layer in layers:
        x = layer(x)
        
    for p in parameters:
        p.grad = None
        
    # calculate loss and apply backprop
    loss = F.cross_entropy(x, y)
    loss.backward()
    lr = 0.001
    
    # update gradients using gradient descent
    for p in parameters:
        p.data += -lr * p.grad
        
    if i == 0:
        print(loss.item())

2.2962050437927246


In [74]:
get_split("val")

3.287923574447632

In [75]:
get_split("train")

2.3637118339538574

In [18]:
# Keeping track of loss details
@torch.no_grad()
def get_split(split):
    if split == "train":
        X, Y = Xtr, Ytr
    elif split == "val":
        X, Y = Xdev, Ydev
    else:
        X, Y = Xte, Yte
    
    # forward pass
    for layer in layers:
        if isinstance(layer, BatchNorm1d):
            layer.training = False

    for layer in layers:
        X = layer(X)
    
    loss = F.cross_entropy(X, Y)
    return loss.item()

In [76]:
# sample from the model

for _ in range(20):
    out = []
    context = [0] * block_size 
    while True:
        x = torch.tensor([context])
        for layer in layers:
            x = layer(x)

        logits = x
        probs = F.softmax(logits, dim=1)
        
        # sample from the distribution
        ix = torch.multinomial(probs, num_samples=1).item()
        # shift the context window and track the samples
        context = context[1:] + [ix]
        out.append(ix)
        # if we sample the special '.' token, break
        if ix == 0:
            break

    print(' '.join(itoc[i] for i in out)) # decode and print the generated word


Em Emadd6 G Am F C F Am F G Am F G F G Esus4 F G Am Am G Am Bb F G Em F G#m Dm Cm A5 Csus4 Ab C7 Gbm Csus4 Am C G Am F G F F Am7 C Gsus4 Esus4 Gbm Eaug G C Dm7 G Am A C4 F7b5 E7 Am F C C Fm Gsus4 Dm7 A5 Am G#m G Em7 Cm Gsus4 Esus4 F9 G Am Bsus Csus4 A C Am C G Esus4 F7 Dmadd9 G Dmadd9 G Am Cm F E Am F Dm C G D Am Em Gbmadd6 D C F C F F Am F Am F G F G F F G Am G G C F G F G Am Dmadd9 C G F Am G F Am G Bb C E5 Am F B Csus4 F7b5 G E7 Bb7 G C G G Bbbm F G B Am G C7 Am D7sus4 G9 G Am F C Am F F9 F Am7 Am Em G Em Dm G Am F F Asus4 G Am7 Emadd6 G Am G G7 D G Am G7 G Am F G F G F F C Em Am F G Am F G Ab F Am F Ebm Bsus F A Bb7 E7 Am F G Am D7 G F Esus4 G Em D Ab G Am F Ab7 F C Csus4 F C Em7 C F C C G Am E5 Eaug F C F Am Am F F G Am F7 F F D7sus4 D C Asus4 C E7 Am F G Am7 F G Am C Ab7sus4 Am Dm7 G F7 F G F G Am E5 Am G G Am G C G F Dmadd9 G Am Gadd9 G Eaug F7b5 C Ab7 F Am F C F G Em Am G F F F Dm Am G5 Am Gbmadd6 G Am D Csus4 C D C7 G7 G F7b5 F G Am F G G F C G Am G D7sus4 G Am7 G Gbm7 Em C A5

# Beatles

C C G Em F G C Am F Am Am7 F Dm7 C7 Em G Em F G C F G C F G C G7 Am7b5 C C G G7sus4 C F C C G7sus4 G7 Dsus2 C#m C C F G C F G C7 C7 G7 F7 C7 Bb F7 F7 C7 F7 Gsus4 Cm Cm G7 Cm G7 C G Am F C G Am E7 F6 G C Caug G G Bbadd9 E C#m A B E E F C G F F Gsus4 G F Dm C C F G Am7 F C C E7 F Dm6 C .
Em F7 F Am C F Bbadd9 G F Bb G F Dm G C C6 Bb Am7b5 Dm7 G7 C C G Am7 F G Am Am F E D D C Cm Bm7 Gsus2 Cm Bb C C Cm E C F Bbadd9 G Am A C F G C F Am Am7 E7 C G Am F C C G F C G G G G7 F C F G C G7 G C Dm F C C Am F Fm G Am Am7 F6 F Dm G C E D D B Em F#dim Dm6 C Am Fm Bm E C F F#dim D7 C Dm G7 C Dm Bb G Dm A F Dm Bm7 C G7 E7 Am G F C G G F Dm C C G G F G G F C C E7 Am G G7 G Dm7 Bb C Am F C F Dm G C G F C G F G C C Am C F F G G G7 F C G G7 G7 C C C E7 G G Dm Fm C G9 Bb F Dm Eb F E D G9 Eb6 C Cm G C C Am D7 F Dm F C D7 F Bb G G C F F G Gsus2 F Am Cm F#dim Bb C Am Bm7 E7 Em G F C G F Dm7 F D7 C Fm G Am F Dm F G C#m G7 G G C G F C C Am E7 G G G7 C Am Am7 F Dm G G C G Am F Am7b5 C C7 G7sus4 G C C C Am F C C G Am F C G7 F C Am F C G F C G F Dm Bbadd9 C#m Gsus4 E F G7 C D7 Gsus2 D7 C Am F Fm D D7 C C F Fm C7 G7 .
F7 F7 Bb C G7 F C F7 F7 C7 G7 C Cm Bm7 G Am Em Fm E7 Em G G6 Bbadd9 F C C G Am F C G G7 F F F G G F C Am G F Am C F G C F D7 C C C F C D7 C G G C C Bbadd9 Am F G Am7 F Bb C Am G F C G C F G G7 F C G F C G G C F C G F Bb G Am7 D C C Bm7 F G7 C F#dim Fm D7 F C C G7 G C C7 E7 Am F Dsus2 G Cm Em F D7 C Bm7 Am F G C F Gsus4 G G7 C C G G F Dsus2 Fm A Em Dm7 C#m G7 G C G Am C C Bm7 F C G F C G F Dm C C F Am C Am C F Bb G Gsus2 B F C G Am C F6 F C G G G F C G Am F Am F G7 C C Am7 C C E7 Am C F Gsus4 G C Am Am C Fm C F F7 G Dm D7 C C E7 Am C Am C C F#dim Dm7 Bb C Dm7 C7 C G Gm7 D G7 C G9 F#m C Am G Em C F F G F C G Bm F Eb6 C C F G F Am7b5 C F Am G G7 C Bm7 D7 G7 C Bbadd9 Em G C F Am C F C G F Bb Gsus4 C F Bbadd9 C C C Am Gm7 C F Gsus4 Dm G7 G7sus4 G B C F Fm C G7 G F Gsus4 G Em G7 C G F#dim Dm C Am C E7 Am F Bbadd9 D7 F C C G F D7 C F Fm D7 Em C G G7 C .
C Bm7 D7 E7 E D Caug C Am Bm7 F C G Gsus2 C C G Am7 C Dsus2 G C Am Am F Dm Bb C Cm Am7b5 G Am Em F G C Dm C C Am Am F C6 Bb C Gm Dm G G7 F C G Am F C G G7 G C F G C F G C Am7b5 F6 F G C G G F C F#dim G F E D D C Am G F F F#dim G7sus4 G7sus4 B E B Caug Fm C Cm Bb F Dm7 C6 C Bm C F6 Eb G7sus4 Gsus2 G Dm F7 Bb C7 F7 G Cm Eb C C Am Am7 F Bb C G7 G C F G C E7 F G C F Dm7 C Am F C G F C C G7sus4 D7 C C Dm F7 C Am Am E7 C G F Dm Eb G7 C F G C C Gsus4 C7 C7 Am F7 C7 E7 Am C F G C F C C G Am7 F C C G G F G C Dm G7 C Dm G F C G Am F C C F Fm D7 F Dm7 G Am F F G F F Bbadd9 F F Em G C .
Bm7 F C Dm7 G Am F C G Am F F G G F C C F C7 Bb C G Cm Dm7 E Gsus2 D C C C C C7 E7 C G F Dm G7 F C C E7 Am C Dm D7 F C Am G F C C G7 D7 C Cm G C F Bm7 G Cm C C Am Am7 F C Bm7 C Am F C7 G C G F Dm F G Am Dm Eb C F F G G F Dm G7 C Eb6 C7 G C F Fm D7 Am F Gm C Bm7 G C C6 Fm Dm7 Em C C Dm C Dm7 Bbadd9 F C G C F Bb F Am Gm F7 F7 F7 Dm7 G7 G7 G7 C G Am F Am C G7sus4 Gm7 C C#m Fm B C F Bbadd9 Am7 F F E F C C F D7 Am7 F Dm C Dm G C F#dim C7 Bb Dm Cm C C G Am F F#dim Cm F C7 G C Am F C G Am7 Em Dsus2 G C G F Am7b5 F G7 C7 Bb C Am F G Em G7 C F6 Em G7 G Am Bm F C Bm7 F Dm7 Bb D7 G7sus4 C#m G7 G C C Em Dm7 G Am7 G7 Dm7 Bm7 C G G F F F7 Fm Dm7 Dm7 C Dm Em C D7 C C E7 C G C Bm7 C C G C F D7 C G G Am7 F C C G C C C E7 Am Am7 D7 Am Em C Bm7 C7 Bb G C Am G7 D7 C G C Am F Dm B F E Gm7 G7 B E D B E C#m A E D G9 C F#dim C C7 G9 Gm Cm F7 F Dm D7 C C G Am F C C G Am7 F F Gsus4 Dm7 F G Em G7 G Am F G C F G C F G C F Dm C Am Am F C G G F C C E7 C G Am E7 C G Dm Bb Bb Bb Dm7 Em Cm Am7b5 F7 C7 Dm7 C7 C7 F7 G Eb6 G7 C Am F C G Am F Am C Bbadd9 F C G F Bb F C G C9 G G Gsus2 D C C Eb6 G7 C Cm Bb Eb F D C Am Bm7 E7 Am Am7 F C Bm7 D7 E7 C7 G C G C Am7 C Bb C Dm Dm C C G F Dm7 G C F G C F G C F G6 Fm Bm7 C7 F#dim C Gsus2 G C Am7 F Am C F Bb D7 Em Dm G Am7 C E7 F C F G9 F C C F D7 F Bbadd9 C F Fm C F G Gsus2 F Eb6 C C G F Am Am7 F G C Am7b5 E7 C G Am Dm7 F G7 G G F C G F F G Dm C C F G G F C F C G G F G C F Bbadd9 F Dm Bbadd9 C C Bm7 E7 Am G A B E C#m F Gsus4 C G D7 F E F E D D C Am F#m F Dm7 G G7 G7 C G D7 C F Fm C Am G F C C G F C G G7 Bb C C Cm Bb G7 G Cm Fm C C G7 G Am7 F C G G F F G7 C C Am F Am7b5 C G G7 F C C G C Bb G F G C C F F7 G F G C F G F Dm F G F G C G7 Fm G C F7 C7 F G C F G C G7 F Dm C Dm C C F Dm6 C A B E C#m A B Dm G F C G G7 G C F Dm7 G7 G7 C F F7 Bb G7 C C Am C F C7 G C F Bbadd9 G7sus4 F C C6 G7 C Gm7 C F#m F Bb C C G F C G Gm7 E C#m D B F C G Am Am F C C C F C G Am F Em G Am Em Am7b5 C7 C7 Bb G7 C Bbadd9 C F Gm F C G F E D Gsus2 A G6 C G G A F A F#m G7 C C Dm F6 C F Am D7 G7 C Dm7 F C7 Am Am Em F G G7 G7 F G F Am Em G E7 F C C G Gm F G C F Dm D7 C Am Am F C G Am F F G D7 C G Am F G C F G G F C C Dm7 C G F F C C Am F F F Dm7 D7 G F Gsus4 Am C F C Am Am7 F C G Am F Dm C C F G F Bb Fm C Cm Bb C Dm7 C7 Dm7 C7 G7 Dm6 C C6 E C F F G C F Am F Dm G7 A G7 C C Bm7 E7 G7 F E F G G7sus4 Gsus2 A C Cm Gm F7 Eb G C Gsus4 Eb6 G D7 Em C Dm7 G F C G Am7 G C C F Bb C7 C7 G7 Cm G7 C Am7 F Bb Eb A C A C C Dm7 G Am7 E7 E C#m G7 Caug Gm Dm Cm C7 G7 Cm Bb C Am C#m G Am F F#dim G F F#dim G C F G C F Dm7 G Am Am F F G C Dm6 C Dm C Dm7 D7 C Dm E7 C G F Am F C G F C G F G C F G Am7 F A C Dm7 Am E7 Dm D F#m F Dm7 C G7 G7 G G G7sus4 Gsus2 D Cm Bb Fm C G7 F7 G C C E7 Am C G F Bb F C D7 G7 D7 C F G G F C Dm Am7b5 C Am C Dm Gm E7 Am F Dm Gm C7 C F G C F Fm D7 A C F G G7 F Fm C C9 F G C F G C G7 F6 F G C E7 Am Am7 C F G F C C Am Am C C F Fm F .
C Dm C7 Bb Fm Dm7 G7 C Am Em Dm C F7 G F7 F7 C6 C7 F7 G Am C G C F C G G7 F7 Gm7 C F Am7b5 G Cm Bb Bb G7 C Am Bm7 E7 C G C G F G7 C Am G C F G C F Am C F G C F Eb C Am7 C C Bm7 Bm7 C C F7 Am D7 C Gsus4 Am Am F C C G Bm7 C D7 G C F G C F Am F Dm G C G C F Fm C Em F7 G Am7 D7 Em C Am E7 F G Am Gsus2 F Bb C C Am F G6 G G7 G C F C G Am G F C C G G7 C G C Am7 C F Dm6 C A F#m Bb C C G G D7 F C C Bbadd9 F Dm E7 C G C F C G7 G E7 G C Am C F G C F Fm G Dm C C G G C C C6 Am7b5 Fm Am Cm Bb Bb Dm G7 C F G C Em G G C G F7 .
C Am F G C Am F Dm7 G F Am Am F C Gsus2 A Bm C G7 Am Am7 F Dsus2 F Dm7 C Dm7 C7 F G7 F#dim G7 C Dm Em Am F F F Dm C Dm G7 C Am F C Am F E F Bbadd9 F Am F Am C C F C D7 D7 C F G7 C F Dm F Dm G Am C F G C Am F F F E C9 Am7b5 Gsus4 C C F Am Am F C C Cm F7 C7 G7 C7 Dm G C Bm7 E7 Am F F G F C G G7 C G C Am C F F Dm B C G Am F F G F F#dim C C Bm7 E7 Am C7 E F Dm7 Am7b5 C Cm G Am Gsus4 G Am C F6 F Bb G F Fm G F C G Gsus2 E7 Am F F G Dm6 Am7 F C Em Am Bm C A C A B C G Am F G G G7 F Caug G7 C Am F C Dm G C Dm C F Bb G C C Bbadd9 C F Bbadd9 C F Am C Am Bm7 E7 F G C C F Bb G Am F F#dim Dm7 G7 G7 C C Am F G C Dm7 G G F F Gm7 G C Dm G C Am G F C Am C F G G9 Gm7 C Bm Cm Bb C Cm C C G7 C D7 G7 C Am F C Am Am C E7 Am F Am C E7 C G Am F Bbadd9 C C F6 G C C F G Bm C F F6 C G F Bb G6 C7 Am G Am C F G G F E Gsus2 F Caug G G C9 F C C F7 E7 Am F Am C G F C6 C C G D7 F C G G Dm7 Am C F G Am G7 Bm7 E7 Am G F C G F C G6 F G Em Dm D7 C C9 Fm C F Dm G C Am C G Am7 F Caug G C G Am F F G C D7 G F C C F Dm G F F#dim G G6 Am7b5 Dm7 C7 C Cm G7 G G7sus4 Em Dsus2 G Em Caug G C F F#dim G Dm6 C F C G F G C Am F C6 C C Dm6 Am7 D B Am7 B E Gsus2 B Gsus4 G Am F C G7sus4 G C C9 C C7 C7 Dm7 G G7 F G Am Am Bm7 E7 Dm7 D7 F C C F G C F G C F#dim G C Bbadd9 Bbadd9 F G F F Dm7 G7 C C Bm7 .
G C F Dm7 G G7 Cm Fm D C7 C F7 F7 C7 C7 C G F Dm7 E Gsus2 C C G F C Dm G C Gm7 C G Am F G C F G C F G F7 G Am Am G G Em Dm C C G C Dm Bb G7 Eb6 C G C9 Am F F G F Am Am F C G C F Dm C .
C G F E D E7 Caug E C#m F C C G C F G F G7 G C C Am Am7 F G G C Fm C Bm C F7 Bm7 C C C7 G7 F#m G7 Dm G C Am G F C Em G F Dm F G C F G C F Bbadd9 C Dm F G C6 G7 C Dm Am7b5 C C Am F Bb F C G F C G Gsus2 C Am C E7 Dm G7 C C F Dm7 C Am F E D B E C Gm7 C G Am F C F6 C C Dm G7 F7 D7 C F C G7 G C F C Am E7 Dsus2 G7sus4 C C#m C G F Fm C G7sus4 G G7 E F#m C9 Dsus2 C F C G F F F G G F C Am G F F F G F F G D7 F G F G F C G F G C G Am F D7 C Gsus2 D F7 F#dim Am Am7 F F Am Dm Am7b5 C C G G Dm G D D Bb C C C F7 C Am Am F C Dm G7 Cm Am7b5 G F C G Am F G Am G F C G G7 F F G F Dm C G7 F G G D G C C Bbadd9 F G C F C G F Dm D7 Dm C F C D7 G7 C Am Am F E D D Em Am C6 A B C F Bm7 C Am G F G Am Am G Am F C C G F C C Bm7 C Am C F Fm C F Dm G C F G D7 F C C G F C C E7 Am F Em Am F G C F Dm C Am G Am Dm F C G Am C G C Dm F G C F Bb C7 G7 C Am F G7 F Cm B C G G7 C F C C C7 D7 C F F#dim G G7 G7 C Am C F C C7 G Bm C C G G7 F6 Am Am F Am C Em G Am F C G F G G C F C Am7 D7 G Am C F G G F C E7 F C9 G7 C Dm7 Bbadd9 Am7 F C F C7 F C F7 E7 Am F F C G F F G Am7 F C Am7 G7 Bb C7 Bb Em .
C Am C F Dm F7 D Gm G C C Am F Am7b5 G Bbadd9 Am C F C C G F C Am F C G Am F G G G7 G C C Fm D7 C Am F C G Am F Am F Am C F G C F Bbadd9 D D C C C F Fm G C F G F Am7b5 C E7 Am F Dm F G7 G C F G Gm7 Fm Caug C Bm D D F#m C Cm G C Am Am C F C G G C D C F F7 F G Gsus4 Bbadd9 C C Gsus4 G Am F Fm Gm Am F F G C F C G Am F G C Am Am F Am C G7 F7 G C Am Em G C F Bbadd9 C Am F C G F G C F Am G Em C C E7 G7 F C G G F C Dm7 C7 Bb C G Em F G C F C F G C F G C G C F G F G F F F Am F F G7sus4 D7 A Em C F G C G G F C G C F G G7 F F G C7 Am7b5 D G Cm Dm C Dm Am7b5 G6 Am G F F Fm D7 F C C G F#m D C C G Am F A G7 A C Caug F#m Cm F Cm C7 Bb G G6 Cm G F Dm C Cm Bbadd9 Am7 F#m F#m F E Eb6 A A F Caug F#m C#m F G D C C7 Gsus4 G G C C Am G F F G G7sus4 G7 Eb F Dm Dm C F G C G Am F C Am7b5 D7 C F Am Am7 F Bb B C G Em F G C F G C F G C F G C G F G C Cm G F F G Bbadd9 D F Bbadd9 C#m F G C Gsus4 F G G7sus4 G7 C C Am Am C C C Dm G F F Eb6 G G7 C Am Em C G F C7 C7 G7 C Em C F G C F Am7b5 C Dm G Am7 G7 C C Am E7 Am C#m F G Am F F G F F Fm G Am Am F F G F G7 Am C C G Am D7 F G C F F C C Bm7 D7 C C G C C F F G D C7 C C C E7 G9 D7 G C6 Bbadd9 F Cm G G Am F C D7 C F Dm G7sus4 D C Am G F C Em C7 G Am Dm Em C D7 F C C G Am C C Bm7 Bbadd9 F F G Caug F E D B G7 Am7 F Bb F G7 D G Gsus2 Am7 C Cm F Dm Cm Dm C G7 G Gm7 F#m D Am7b5 C Bm C Am F Dm C C F G F E G7sus4 G7 C Am F#m F Bb G Am Dm G9 Bb C G F C G F Dm7 G Am F C Gm7 C G Am F C C Am D7 F C C Bm7 Fm C G F C G G7 C C F G F Dm C9 Cm G F G7 Am F C G Am F Dm G G7 G C Am Am7 E7 E D D C Dm Eb6 Bb C D Em Dm C C F G G7 Dm G E C F F G F F Bbadd9 C Am Em Fm D Gm Eb C#m C F Bbadd9 C F G C G Am E7 Am F F G C C C C E7 C G F F6 C F C7 C G F7 Dm Bbadd9 Am7 F E F E Em G F Dm C F Bm7 C7 Dm C C7 F F G D7 G7 C Dm7 Am7b5 Dm7 Bb C Bb C Bb Am7b5 F#m F C C Am F Bb C7 Am Em F7 F Am Am7 F Gm D7 Gm C7 Bbadd9 Am7 F C Cm G C F Am Am7 F Em F G C F Am F Dm7 F Bb G7 C Bm7 Bm7 D7 C G Cm Em C Dm G C Am F F C F G7 F C C Dm6 F G G7 F Dm C Bm7 E7 Am F F G C Dsus2 G C F7 Bbadd9 E7 F G9 G Cm C6 G7 E F Dm Am7b5 C Bm7 Bm7 E7 Am Cm C C F Dm C C G F Am F Dm C Eb Em C Am F G7 G Am Am7 F C G F G C F Am C Dm7 F Am Am Em C C G7 E7 E C#m A E C Dm C Bb G F C Am F Fm G F Am F Dm7 G7 C C G D7 F C D7 C F G6 G G7 F E D D C Am7 Em Bb C#m C Dm G F Eb6 F F C7 Dm7 C G Cm F7 F6 C7 Bb G G7 G F C C7 D7 C Am C G7 C C Bm7 G7 C F6 Am G7 F C C Eb6 F#dim F C G F F C Bb F Dm7 C9 G Cm C7 C7 Dm7 C Am7 E7 E Am7 A B F#m B E C#m E C#m A F#m G7 F G7 G7sus4 Am A C A C C Am F F#dim C7 G G7 C G F G7 C Dm G C Caug Bb Bb C Cm G C F#dim Am Am7 F C G G F Fm G G7 F C C G7 Am7b5 Bb C F7 Em C7 C7 C7 F7 C7 G7 C D7 C F F#dim F7 Am7b5 G7 F G Dm6 G7 F Gsus4 C G G7 D7 C Dm G7sus4 Eb A C#m A C#m G C C Bm7 Dsus2 C C Gsus4 Fm D7 F C G Cm F G Gsus2 G7 C Dm C G6 Bm7 G G7 F Dm7 Am7b5 Cm Cm G7 G7 F C Dm Eb Em G Em G C A C C Bm7 E7 Am G F C G D A C Cm Em G F G7 C Am Am7 E7 Am F G C F G7 C F F G F F G Eb6 G C C C C E7 Am G Em C Am Am F C F G Bbadd9 Am C Dm Bb Dm7 C C Bbadd9 Am F F C G Am Am7 F C G F#m F Am G F Gm7 Bbadd9 C F6 Bbadd9 C A E7 C#m G Gm7 C F Cm G7 D C C Cm C G Cm C7 G7 C7 F7 G G7 F Am C F G Am F C Bm7 Bb D7 G Am Bm7 F G7 G C F G7 G7 F F Am D7 Am C F C G Em F#dim Bb C G7 Am C G7 Em E7 Dm E7 Caug G F F Dsus2 D C C6 C Dm7 G C C7 Bb G Em C6 G7 Bb Gsus2 F7 C G Am7 F Dm7 C7 Am7b5 C7 G7 C Am Bm7 C E7 G G Bbadd9 D D E C F G C Am Am F C G F C G C Bm7 Bm7 C Dm G Am7 Em C G Em F G C D7 G A Gm7 E C A E F E F#m C#m G G C C Bm7 E7 Am G C Am F C G C C Dm G C Dm F Bb Am7b5 G G C Am Am F E C G F G Am7 C Am Am Am Am7 C F Am C Am F G C Am C F Bbadd9 F E D D C G C C Am Am C F Fm Am7 G7 F Dm C Dm7 Am Am7 C Gsus4 F D7 C F Eb6 G7 C Dm7 Am F Dm7 F F#dim G C F Am F Em G F C G C9 F C G G7 G F Fm C C7 Eb Em Am D Bbadd9 C C F Am C Fm C Am7 G B C G Am F Bm7 Bb F Dm7 Cm Bb Fm E Am7 A C C Dm6 C E F F B F G Gsus2 Am C G F D7 Dm7 C Am F Dm C F C D7 D7 C G F C Dsus2 F Dm7 Em Dm Eb C Caug Am7b5 Bb Am7b5 C Am F C G F Dm6 C F Am C Dm Bb C C Bm7 E7 Am G C Bm7 E7 Am G Am F C G D7 F A G F C G G7 C F Bb Bb Am .
C G G F Eb G C G Dm7 Bb C F7 C C C6 Am7b5 G C F Gsus4 Dm Eb6 A D Eb C Cm G C Am Dm6 F A D A C C F Am C F G Am F G C Am Am Am7 F Dsus2 C F E G7sus4 C#m E C#m E E Gsus2 B A B E C#m Gm7 C#m G F#m F E F#m D Caug A C Dm Em C7 E C F Fm G Am G Am A F C Dm G C Dm F Bb C7 C Bm7 F6 G7 C F C G Em G7 Am D F#m C G G F Am G7sus4 F Bb Dm Am D C Cm F7 C7 F F7 Bb G7 C Am Am7 C F Am D7 G7 C Fm G9 F7 C F7 G7 G7 C Am F Bbadd9 C F Bbadd9 C F E G7sus4 F F#m C Eb6 G7 C Am G F C G C C7 Bbadd9 C F F G C F G C G Am F G C F G C F Gsus4 Fm G F Dm C C Dm6 C Dsus2 E Eb6 Dm Gsus4 C C7 Caug G7 F Cm C Am D7 C G G F C F G Am F G C F Bbadd9 Gm Dm7 C C G F C C F Dm C C Am E7 Am G F C C Am7 Gm7 G7 C#m C G Am Am7 F C C G F Dm G7 G7 C F Dm6 Em F G7 G7 C F C G G7 F C C Am F C C Bbadd9 C F Dsus2 Em G7 C7 Bb G7 C Cm Bb C7 Dm7 G G Am C Bm7 E7 Am Am7 F C G Cm G9 G Gm7 Bm F G C Am Am F F C Am F Dm F Dm C F F7 C7 G Am C F G C F Dm7 G Gm7 E D F#m A C F Am C E7 F F F G7sus4 D D G9 Bm7 G Bm F G F#m D7 C F Am G7sus4 F F Bbadd9 C F E F C G Am Dm7 Bbadd9 F F Fm C Am Am F C Fm F C C E7 Am G F Dm G G7 G C#m C Gsus4 Am F Dm G C F G G F C Am F F Fm G Am F G7 C C7 G7 C G Am D G C Cm Bb Bb G7 G Dm G C Cm Dm G G7 F C C Am F Dm G C C F G C F G C C7 Fm G7 C Am F Am F C G D7 F C G D7 C G C G Bm7 Bm7 C C7 Dm C G Cm Bb Gm G7 G7 C Dm D7 C C Bm7 D7 C Dm7 Am F C G C F G C F Fm C7 G7 D F Bb C Am C Am Am7 F C G Bbadd9 Am D7 Em C G G7 F G G Em G9 G6 Dm G F Gsus4 Bm D7 G C#m F F G Bb G C G7sus4 G A Gsus2 A Em F E C G C F G C Am Am F C G G7 G7 G C G C#m Dm7 C Am Em Am Am Am F Am7b5 G G7 F C G Am C F Cm Bb E F#m D F#m D G Am C7 C F7 F7 C G G Am7 F6 C Em Am F Dm C C Am F Am Bm F C C D G7 C G9 Dm7 Bb G Am F F Dm7 C7 Bb C G G Am G F C G F C Am G F F F G Gsus2 F C C Dm C Am7 Bbadd9 G F Gsus4 F D7 C D G7sus4 Em C7 C G Am F C G Gm7 C F Gsus4 G G C E F C6 E C Am F G C F Am Bm7 Dm Am D A E C Bbadd9 F Dm G F Dm C F F#dim Bb Gsus2 D C6 Bb G Caug Em C Dm G7 Bb C G Am C C E7 Bbadd9 C F Bbadd9 Am7 A C C Am Am7 F C C G Am F Dm C C G C G Am F C G G7 F F G C F G C F Bbadd9 C F Bbadd9 C F Am G F C C G F Dm C C7 F7 G6 Dm C Am Cm Bb F7 C Cm C7 F7 F7 Bm7 C7 G Am Am E7 C G F F C G E7 G Fm F F6 C Am F Bb F6 F Dm C C G7 F G C Am F Bb D7 C Am Am7 E7 C G F Am C F Bb Bb F7 Cm C7 Dm G7 C Bm C9 E C#m F E D D C Am F Dm G C C Bbadd9 F C C Bbadd9 C F Bbadd9 C D Em Bb B G7 G F Dm7 C Am Em Fm Bbadd9 F G7sus4 C Am Bm7 E7 Am F Am C F Dm G F C C G C9 F G C F Am C Am C E7 Am F Dm G7 Dm7 Am7b5 D7 Am F F Gm7 C F Bb G G7 Dm E A A C C F G F Am G F G C F Am G7 D Eb C Dm7 F Dm G C C Am Bm7 G7 C F Am G F A C C#m C G C G G F F Bbadd9 F Caug Gm Em Am C G F Dm7 C G Am G F G C Am7 E7 Dsus2 G Gsus2 G C F Bbadd9 F Cm Am7b5 C7 Bb C Am F C C Am E7 F G C F G C G F C C F Am C Am F G F G C Dm G7 D F#m C F7 F7 C C7 F7 Am F7 F G C G7 Am F C C G C C Am7 F C7 Bb G7 Cm F C C G Am F G C F Am C F G C F Bbadd9 F A E C Em F G G F F Gm7 C G7 F C G G7sus4 G7 A B E C D Bb G7 C C C Dm G7 Dm7 F C Am Cm Dm7 Gsus4 G C F C G Gsus2 F C G Bbadd9 G F C Dm7 Gsus2 F G7 G D F#m C G F C Am F G C Am Em Am G F Dm F Dm7 G C9 F F Dm Am F Dm C Am F Bb C G7 G Am7 Em C C#m Am A D B C F F#dim G F C G G7 G F C G F G C F Fm C Am F F Bb G7 C Am Am7 F C Bm7 E7 Dm7 G F G C F G Am7 G7 C F Fm G7sus4 C F7 G Am Am7 F E Em Dsus2 B C F C G G7 F F Dm Gsus2 Eb C C G Am Am7 F E D D C G F F6 C G G C Em Bbadd9 Am7 F C G Em G7 G F C C G F C G F Dm C F Fm G F Fm F F G G7 F G C F C G G F C C Em Am G F C G Am Am C G Gm7 C C Em G7 Am7 Cm Bb Eb6 C9 Gm Dm6 G7 F E F G C Am F Dm C F G6 G F7 C7 C7 Bb Gm D7 C F7 C7 C6 G7 D Bm7 C C G F C G G7 F Bb Am7b5 F C G Am Am7 D Fm Cm C G F G G G C Am C C Am F C G G7sus4 F C D G7 C Em Am Am Am F C C G D7 F C G Am F Dm7 Bb E C Dm C C G Am7 E7 E C6 Caug E C#m F#m B E C#m C G Bbadd9 C F G C Am F Dm A Am7b5 F Am G G7 C G Gsus2 G9 C C G E7 F G F F Dm G Cm Am7b5 C Dm Dm7 C7 G7 C G F Dm C Caug G C Gm F7 F7 Am Bm7 Gm7 C G Am F C F#m G F C C G Am F Dm7 G Dm7 Am7b5 G C G Am7 B C Dm6 G7 B E D B E Dm6 Gsus2 F#m Gm C G F C G Am Am7 F#m C C F G C F Bbadd9 C Dm7 Em Dm G Dm7 Am7b5 Bb G7 C F#m F Dm7 G F C G F Dm D7 C#m C#m F G C Am Em Dm7 F C7 G Am F Am F G C Am F C G F G C G Am F Fm G7 G7 F Fm C D7 F G C F G C F G C Am F C G D7 Fm F Am F Am G F C G G7 Eb6 Bb C Cm Am7b5 F7 G7 Em F7 D7 C F F6 D7 D B C G Am F E C#m E C#m Caug Bb E C#m F G9 E7 C G Dm Am F F#dim E D C C Am G7 Am7b5 G C G C F G G C F G C F G C Am Em Dm Bb C Dm7 G7 C D7 G F E D G C Dm Eb Bb A B C#m Dm6 D D G F7 G C Dm7 D7 C C Am G7 D7 F Dm F Dm F G C#m F F C G Em Am Am F C F Bbadd9 C Am Am7 E7 A F E Em Am Em G C F G Am F Bbadd9 G7sus4 F C Cm C7 G C F Bbadd9 C#m G7 C Dm7 G C F F G Bm7 F Bb Bb G7 Em G7 D7 C Dm G F Fm G Dm F .
C G D7 C Dm6 Am F F G F G C Am C Dm C Am7 Gm7 E F#m B C Dm C C Cm C C F#dim G D7 F E D Caug C C Em G C Am F7 C C E7 Am F Bm G7 C A F Bb G E7 F F C C Bm F Dm F#dim C F Gsus4 Fm G Am Am7 F C C G G Am7 F Dm7 G7sus4 G C C Am F Gsus4 G F G7 Gsus2 G C Gsus4 F Dm C Am Am F C Am D7 F G Am Am C Am F G C Bbadd9 Am Dm7 C Am Am F G C Am Am7 C Am F Bb C Am Am7 F Fm C C F Am C C F7 C F G7 C C C7 C F7 C C G F G G7sus4 F G B C Fm D7 C F C G Cm Bb Eb C#m E G7 B C E Em E D D C F F G F G7 D7 G7 Dm G7 C Am E7 Am Bm F C G Am F C Bb Bb C C G C F F Bbadd9 D7 F Bbadd9 F Dm Bbadd9 Caug Am7b5 C A G7 F C F G C Am F C Dm7 G7 G C G G C G C Dm Gsus2 B G7 C C F G D7 F C C G C C D7 Am7 C7 Dm7 Bm7 Cm G7 C7 Dm Am F7 Fm E7 Am G G C G C F G C F Bm7 C C G C F F#dim .
F7 F#dim Gsus4 G Dm C C Am Dm7 G F Dm C G D B C Am F F G C F Bbadd9 G Em Bbadd9 Am7 F C Em G7 C7 F6 C Am Am F C C Am Am F C G F E C#m F#dim Caug C Am Am7 F Am G Am C C C Am E7 G Dsus2 G C Em Dm G C C Am Cm C C G G C F Fm C Am Am7 C C Am C Am Am F F Dm7 G G7sus4 F C Dm C G Bbadd9 F G Em G D G C G Am F Em G Gsus2 Em C G C D G Am C F G C F G F F G7 G G C F G C F C G Am7 F Dm C G7 Em C F G C D Fm C C Bm7 Em Am C Dm7 Em Am Cm Bb E F E C#m Am D E7 Am C Am F Bb C Dm C Dm7 Gsus4 Am G7sus4 G7sus4 B E Cm A G7sus4 Cm C Cm Gm G G7sus4 C7 Bb Bb G7 Am Am F Fm C Bm7 Gsus4 F C D7 D7 F C C G Am F Dm7 G Caug Bb G7 Bm7 F6 F C G Em C G C F G C Am F G C Am Am F F6 Fm C F G C Am C E7 Bbadd9 D7 F F C G Am F Dm F G Cm G7 C Cm F7 Eb D D C C C G F C G F#dim Bb G7 C Eb6 Bb G7 Bm7 F#m F7 C7 Cm G G7 C7 Bb G7 G Cm C C C7 G F Dm C Dm7 Am7b5 C Dsus2 G F F#dim Dm C G E G7 A B E C#m A B E B A F#m D Bb F C F#m G Am C F F G C F Eb6 F G7 Cm Dm7 .
Gsus2 C C F7 Bb Dm7 Bb Cm C7 Eb6 G7 G7 G7 C Am G F C G G7 F Bbadd9 G F Am G Am F Am G Gsus2 F C F C G Am F Dm7 G9 F Am G G7 C G F Em G D7 G7 C F C G C G Am C Bbadd9 C F C G D7 F Am F Am C G G F C G G7 C Dsus2 G C Am F Fm C C Bm7 E7 Am G Am F Gsus4 Dm G7 C Am Am Bm7 E7 Bb Eb D D F#m C C G Am F Fm Bb G7 C Am E7 Am G Am Am7 B E C Am7b5 E7 Am F Am D7 F G Gsus4 G G F Bb G7sus4 C Dm Am Am Eb6 C6 Bb C Cm G7 C Cm G Em F E F C C G C F G G7sus4 F G6 F C C9 G F C G Am C Gm7 C F C G A G F G F G C F Fm F G7 Am7 C F C Am F G Em Dm G G7 F Am C F C G F E D7 Caug E C G Bb G7 C Cm C G Am F Bbadd9 C F D7 D7 F C Am G F C G C F C G F G D F7 F7 Cm Am Am Bm7 C C Dm F G F F#dim G G7 F C G F G7 G Dm C C F G C C Am Am Dm C G Bbadd9 Am7 F6 E C#m Dm F#dim C C G C F Fm C Cm G C F C G Am Am7 Em Bb F G7 Gm7 C F C G G D7 Am E7 C G F Fm C C D7 C C E7 C G6 Dm G7 C G9 E7 Fm F G7 F G C Am Am F Fm C Am G F F G G F Fm G7 G7 Em Am G F G C F G E7 Cm G Am F E G F G C G Am G7 C C Dm7 C9 Am G F Caug G7 Cm F F7 C7 C7 Dm7 Am Gsus2 C C G G7 G7 C C7 G C G C E7 Am G G7 E7 G7 Em Dsus2 F7 G Eb6 G C C Bm7 Am F G Eb6 Bb E F Dm Am7b5 F Caug Bb C Am Em Am Am G C Caug G7 B Dm7 Bb Am7b5 C Am G A C C Cm C7 Dm G7 C Cm C7 F7 C7 F7 C7 C Gsus4 C G G7 F Dm C Dm G C F G C Am C C G Am C F G F Am G Gsus2 F Dm7 Eb C F Em Fm G G7 Am F#m G C Am C F7 F Am C F G C F G C F C C Am F Fm Gm F G7sus4 C Caug G C F G C Am F G7 C Am Cm Bb Bb F C C7 Dm Fm G7 C Bb Bb Dsus2 G7 F G F6 F G Am7 Dm6 C F Bbadd9 F C G F C G C6 Bb Dm6 C#m F E D D G9 Bb E Gm Em Bbadd9 F Dm7 Am Am F Dm7 F C C F Am C C F Dm C Fm F C C G F Dm G G7 F C G F Fm Am7b5 G7 C F Dm7 G7sus4 G7 Bm C F Am G Am G F G C G C G Am Am7 D D C C Am C F G F C G Em G G Gm7 Gsus2 F C Eb G7 F E Em Dm C C G9 Am F G C F G G Am C#m Am C F Bb C7 Bb G C F G G F F Dm G Dm6 Am7 C E G7sus4 F E D D C C F7 C7 E7 C F G7 G Dm6 C F Bbadd9 Am F Fm C E7 F#dim G C F F G C Am C E7 G D Em Am7b5 F G7 C F C C Am C C C G G F C G F C G G C F G C F G7sus4 F G7 G C Dm Bm7 C C C E7 Dm Am C Am C G7sus4 E7 C G D7 C C Dm6 G C G F Fm D7 G F C G Em C C G E7 Fm C Dm C Dm Em Dm G C Am Am F F#dim Bb C G7 Gm Cm Bbadd9 F G C#m C F Am C E7 Am G Bm7 F G Am7 F C G G7 C C G7sus4 C Dm Cm Dm7 F7 G7 Am7 C D7 C C F Dm7 Gm C9 G7 G C Am F C C G Am F C G Cm G7 C Em C C C7 E Am7 A F#m F#m C Dm7 G C G C F F G C A G C9 C G Am F C G G C E D D C G F C Dm G7 G7 C C C Fm Eb C C C G7 C C C C Am F Am C F C G G C C G Am Gm F Am D7 Bm7 E7 Am F F C C Am F C G F E F Dsus2 Dm6 C Em Am F G C F G C F Am Bm .
C F C C Am F Dsus2 D A G C G C Fm Bb F7 E7 E D7 Gm Bm7 E7 Am G7 C Am7 G D F Dm7 C G Am E7 F G C F Em C D7 C F C C7 Eb C F G Dm6 C C Am F F G Dm6 D B E Dsus2 D F#m F#m C Am Am Am F C G G F A C Dm7 Eb C F Bbadd9 Am7 F E C#m F G C F Am G F E D D C C Dm C Cm B Am C F Am G F Am C Am Am G7 F C G F Am C Am F F Bb C C Cm F F7 C7 Bb C Am G F Am F C G C F Bbadd9 C F F G C9 Am F F G G C G C C C F C7 G Gsus4 Dm Gsus4 C7 G C Am C G G F C Dm7 G7 Am C F Dm C G Am F C G G7 F C G F Dm Am C F G C D Am Em C Am G7 F Fm C C F F7 C7 Dm C G G F C Bm7 E7 Am F Em G G7 F C C G6 F6 C Dm F Bb E7 C F F G F Am7b5 B C G G7 F#dim G C Am F Gm7 Em G G F E D G7 F#m G6 Bb C F Cm G G7 D F#m C C G G F E D D C C G Am F C C Am F G C F Dm G F Dm C Am F Fm C Am G7 F Am7b5 Gm Fm E7 Am C Am Am F C G F C Gsus4 G Am Am7 D C#m Dm Em C G Am F C C G F G C G G F C G Am F C G G7 G7 C Dm G7 C G Bm7 G7 F F7 G F Dm C Dm7 G Am F#m C Bb F C F G G F F G C F F#dim G G7 F C D G7 Am E F C G F Am F G7 G G C F G C F G7 G7 Dm6 F G G F E F#m D F#m C C Bm7 Bm7 G F Am C F C D7 Am C G F G C F G C A C F Am C Am E7 Bb F7 C Cm F7 Cm C7 G G F E D A C C G Am F Am7b5 C F Dm C F C G Am F Dm C D7 C Bb G7 C Dm7 G Cm G7 G7 G7 C Eb6 D7 Am C Gm7 C G A G7 F#m D B E C#m Bbadd9 F#m F E Bm7 Bb C Dm C F G C C Am C F G D7 E D D C C G Am F Gsus4 Bb G7 C Am F Dm G C#m Caug F Am7b5 F7 C Am F Dm7 G G7 F C C Em F#dim G C Dm Em Dm7 D7 C Am C E7 Am F Gsus4 G G7 Am Am F C Bbadd9 G7 F C C G C F Am Dm7 Am7b5 G7 C G C F Am C F Bb G F C F Dm F Dm7 Dm7 Am7b5 C7 F7 C7 Gm C7 G7 C7 G Am Dm F Bb Em Am Am E7 Am7 F C Dm6 Gsus2 D C Cm C G7 Em Bb G C C C F7 Bbadd9 G7 C C Am C G7 C Dm7 D7 C G F F6 G7 C C F G C Cm G C G D7 F G C Am F C G F Dm C G7 C C Am C Dm Am Am7 Am7 D B E F#m C Eb6 C Dm Gm C F Am C F G G7 F G C G F C#m C F Gm7 C F C G Am7 C Dm7 G Am C F G D7 F Fm G G7 F C F Dm7 G7 G7 G F F G F Bb Em G7 D Am Dsus2 Dm6 C G9 Am7b5 G7sus4 F Dm7 C Cm Gm7 G C#m E D D C Cm Cm F C G F C G F G7 F G7 G G7 C C Am F Am C F G C F#dim G F F F G F G F Dm Bb G7 C#m Am G F Dm7 G9 Bb C C7 Dm C G .
Bm7 F Dm Em Am Am F G Am Am F C G C Bb G C G7 G7 C E7 Am F F#dim Am7b5 E7 F F F Bm Am Am7 D Gsus2 F#m C F F#dim G C Cm G F7 F G C Am C F G C F G C F C Fm G Em Am C Am F G C F Fm C C7 G Am Am7 F C G F C F Cm C G7 D7 Am C F C7 G C F Bbadd9 C Am F F G7 D7 Dm7 C C G6 G F G G F C C G C F Bbadd9 C Am C G C F G Dm6 F Dm7 Dm6 C C G C F#m F Dsus2 C Am C E7 Am F F G C F Em G F G C C#m Am F Fm G F F7 Fm C7 Bb C G F Fm Dsus2 D C Cm Cm Bb G Dm7 F C G F F C Em G D7 F F C G Gsus2 D D7 C G F C F Fm C Am Am7 Gsus2 C C G F C C G C F G Am7 G7 F C G D7 F F Fm G Am C E7 Am D F Bb .
C Am G F C G F Dm7 Gm C G Am C C E7 G G Dm B F G C Dm F G Am C C G Am Dm6 F G7 G C F Gsus4 G D F C F Dm C F C Dm G7 Bb A G7 D7 F Dm7 Bb D7 D G C G C F Gm G C F G C F G C C7 C7 Bb G C Fm G C C C C Dm G7 Bb C Am E7 F Gsus4 Am F F G C F G Bm F C G Dm6 C F Bbadd9 C F Bbadd9 C F Fm G Cm Dsus2 C Dm Em G F C7 Dm C Gm E7 Am F Dm G7 C C Bm7 Bb Bb C Am F G C F G G F Fm B C G Am C F C C F C C Dm G7 C C6 C7 Bb Cm Eb Caug A C Dsus2 C Dsus2 B E A D A C F F G Cm F#dim C C E7 Am C Caug Am7b5 F Dm C C Am7 G7 F C C G D7 G F E D F C F C7 G7 C G F Bb Bb E Am7 B E C#m Caug Gm C D7 C7 Bb C Dm Em Dm6 C F Bbadd9 G7sus4 Em Bb C Bb Bbadd9 Am7 C A C B Cm G9 Gm Gm Am C F C Dm G Am7 F C G G G7 F Am C F C C Am G F C Am7 D7 F Dm C F Am C G7 Bbadd9 D7 C6 C C F G C Am7 F Dm C Dm G C Am7b5 C F Bm7 C9 Am F G C F Bbadd9 C F Gsus4 Bb G7 F Am G Am F C G Am Bm7 F F G C9 Dm E C#m E C F Bbadd9 F F Dm7 G Am C F G Am Am F E F Dm E G7sus4 C Dsus2 C F#m B C C Am D7 Am F Fm D7 G7 G C G7 Bm7 Bm7 C G6 C7 G C G Em Am G F Eb F Am C Am F Dm C C G7 D7 C F Fm C G Cm Fm C C7 Am7b5 G F Dm C C G F F C G F C F G7 D7 G7 Bm7 Gm Cm F7 Bb G7 G Am Em Fm E7 Am Am7 F C G G F C G G7 C C G Em C C G7 C F G C F G C F G Am7 D F C Cm G C Dm C F Bb D7 Am G7 F Am F Dm C C Am C G G F C Bbadd9 Am G F C Gsus4 Dm7 G7 D7 F Fm C F Dm6 C F .
Bm7 C Cm C Am Am7 F F Am F G C F Fm Dm6 G7 C F Am F Dm C G F C G Am F Fm F Dm C Am G F Fm F Cm .
C G Am C Am F C Gsus4 Bbadd9 C F G Am F F G F G C Am G7sus4 C C G F C9 Am G Am F F G F D7 C#m C F G C F F#dim G F F E D G7 C C G Am Am Bm F C C#m G7 D G C Em Am D7 F Bbadd9 F G C F G C F Fm Eb G7 G Am C E7 Am G F Dm G C C G Am F C C Dm G7 C G F#dim C Gsus4 G9 Bb G7 Dm6 Em F Fm C F G Dm G7 F C G Am F C G Am G C F G C F F F Am Am F Bbadd9 Em F Fm D7 G7 F Fm C F Dm C7 G7 Cm Bb G7 G9 F7 C7 C6 Bb C Am Am F Caug Fm C Am F Bb F Dm7 G Am F Bb G Am Gsus2 C C7 C7 C7 C7 D7 .
C Dm Gsus4 Dm7 Dm7 C7 Dm7 C7 Bb C Am E7 C G C C Am C Dm7 F7 G7 G7 F G G F C G Am C Em C C G F Bb C Am G F C G Em F7 Bbadd9 G D7 F C G Am7 F G9 C Cm G C F G C Gsus2 F Am Am F E C#m F E D C9 C C G Am F Am7 C F Dm G C Am Am E7 Am D G7 C G6 .
​


# Taylor Swift
C Fadd9 C Am7 G Fadd9 C G Am F Bm C Gsus4 C Fadd9 G Fsus2 C G F Am G F C G Am7 Fadd9 C G Am F C Csus4 G F C G Dm C G Am F C G F C G Am C F C G Am F G C G Am F C G Dm F G C G Am F C G F Fadd9 G Dm F C G Am F C G Dm F C G Dm F G C G Fadd9 G Fsus2 Am G F Am G F G Dm C F G C Dm C F Am G F G G7sus4 F C G Dm F C Bbadd9 F G C G F C G Dm F C G Am F Bb G Am Bbadd9 C G Am F C G F Dm F Csus2 G F Csus4 G C G5 G Gsus4 C G Dm F C G Dm F C Am F C G F G C Dm F G C G Am F C D Am7 G Am F G Am G F C G Dm F F C G F C G F G G Dm Fadd9 F C G Dm F C G F Am F C G Dm F Bm C F G Am F C G Dm C F Am G F C G C G Am7 Fadd9 C C Am7 Gsus4 Gsus4 C G Am7 G Dm F C G Dm F C G Dm F C G C C Am7 G Fadd9 C C Fsus2 Fadd9 G C C G Am7 G Am F C F Am G7sus4 F C G C Bbadd9 Dm F A Gm F Am C Am F G G F C G Am F C G Am F C G Dm F F G C G Dm F C G Dm F C G Am F C G Am G F C C Am7 Fadd9 C Gsus4 C Fadd9 G Fadd9 G Dm F G C Am F C G Am F C G Dm7 F Am C F Gm C G Am F C Am G F Am G Fsus2 G G Dm Bm F C G Dm F Am G F C G F C G Csus4 G F G Am F C G Am Gsus4 G F F Am Csus4 G C C Am7 G Fsus2 G Am F C C C Am F C Dm F C G Am F C Am F G F C G Am F C G Am F C G Dm F G G C C C Am7 Fadd9 G Fsus2 F Fadd9 G Am F C G F C G Am F C G Am7 G Fadd9 C Am7 G Fadd9 C Am7 G Fadd9 G F C C Dm7 C Am7 G Fadd9 Gsus4 G Gsus4 C G Am F C G Dm F Csus4 G F Dm G F Am G F C G Dm F F G C G Dm F C C F Am7 G C G G7sus4 F C G Am F Dm C F C G Fsus2 D F Am Gsus4 F C C Am7 Fadd9 C Csus4 G Fsus2 G7sus4 F F G Fsus2 Am C Bm Csus4 G Dm F G C F G F G C C Am7 Fadd9 Bbadd9 C Am F C G Am F C G Dm F Am G F G Am G Dm F F G C Am F G Am G F C G C Am7 Fadd9 C C Am7 Fadd9 C Am7 G Am F C G Am F C G Dm F C G Am F C G Am7 Fadd9 C Am Fadd9 G C Am F Am7 G F F C G C G Dm F G C Gsus4 Fadd9 C Gsus4 C G Dm F G C Am7 G Fadd9 C Fadd9 G Am7 Fadd9 Fadd9 G Fadd9 C G Fadd9 C Am7 G Fadd9 G Dm C F Am G F Bm Am F C Am F C G Dm F Am G C C Am7 G Fadd9 C Fadd9 Am7 Csus4 C Am7 Gsus4 Fadd9 G Fadd9 Fadd9 C Am7 G Fadd9 C G F C G Dm F C G Am Fadd9 F C Dm7 F C G Am F G F Am G F C G Am F G G Dm F Bm G F Am G C F C G Am F G C G Am G F Am G F F G Gsus4 G Dm F C G Dm F Am Am F Am F G Am F C Gsus4 G Fadd9 C Gsus4 Am7 G Gsus4 Gsus4 C G Am7 G G Am F C G Dm F C C G Bm C F G Em Gsus4 G C C D Fadd9 G .

Em Am7 Fadd9 C Am7 G Am7 C C G Fadd9 C C G Am F C G Dm F C G Dm F F Am C F Am C C A Fadd9 Gsus4 G Fsus2 Am F C G C F C Em G Am F C C Dm Fadd9 C C Am7 Fadd9 C C Fadd9 Fadd9 Am7 G Fsus2 C G Am7 G Fadd9 Am7 C Gsus4 Am7 Fadd9 G C C Am7 G Fadd9 C Am7 G Fadd9 C Dm Csus4 F C F G F Fsus2 G C Gsus4 Fadd9 G Am F C G Am F G C G Fsus2 C G Dm F G C G Am F C G Dm Csus2 G C G Am F C G Dm F G C C Gsus4 C G F C G C Fsus2 F Am C G Dm F F C G F C G F C G C F C Gsus4 C Am7 G Fadd9 C Fsus2 C Am7 G Fadd9 Fadd9 Am7 G Fadd9 C G Am7 F C G Am F C C Am7 Fadd9 Fadd9 Gsus4 Fadd9 C G Am7 G Fadd9 C G Am7 F C G F F C G Dm F Bm C G Am F F Am G F G G Dm F C G Am C F Am F C F Am F C Bbadd9 A G Dm F C G Dm F G C G Am F C Bm C F Am G F C G Am F C G Am G Bm F C G Dm F C G Am F C G C Am7 Fadd9 C Fsus2 Am7 G Fadd9 Am C C Am7 G Fadd9 G7sus4 F G G Dm F C G Dm F C G C Em Fadd9 Am7 Fadd9 Fadd9 G C Am7 Fadd9 C Gm C G F Am D C G Dm F G C C Am7 Fadd9 C G Em Am7 G Fadd9 C Am7 G Fadd9 C Am7 G Fadd9 C Fadd9 G Am7 G Fadd9 C G Fadd9 C Am7 G5 C F Fadd9 G Dm C F Am G F Am G C D Dm F G C G Am7 G Fadd9 C Gsus4 C Gsus4 F C C Am7 G C G Am F G C G Dm F C Bb G Fadd9 C C Am F C G Dm F G A F C G F Am C C C Am7 Fadd9 Fadd9 G Fadd9 C Fadd9 G Fadd9 C Gsus4 G C G Am F C G Fsus2 C Am7 G Fadd9 C Am Fadd9 G Am F G C C G Fadd9 G F Fsus2 G G Dm F G C G Am F C G Am F C G Am F G C Am F G Bm F Dm G7sus4 F C G C Am7 Fadd9 G Fadd9 Fadd9 Fadd9 G C Fsus2 Am7 Fadd9 C G C Fsus2 G Fadd9 C Gsus4 Am G F C Am F C G Dm F F Am G F C G A G Dm F C D Am F G C G Dm F G C G Dm F Dm F F G D F Am G F G C G G Dm F G C Am G F Am G C Dm F Am F C G Am F C G C Fadd9 Am7 G F Am G F C G Dm F G C G Am F C Gsus4 C F Am G F C G Dm F G C G Am F C Bb F F C G Am F C G F C G Am F C C F C Am7 G Fadd9 C Gsus4 C Fsus2 Am F C F C Am G F A G Am F C G Am F C G Am F C C G Am7 Am7 C G Dm F C G Dm Am F Am F G G Fadd9 F Am G F Am G F C G Dm F C Am7 G Fadd9 C Gsus4 Am C Am7 G Fadd9 Am7 Fadd9 C C Am7 Fadd9 Gsus4 C G Dm F C G C Am F F C G Dm F G C G Am7 G Am G F G G7sus4 F C G Dm F C G Dm F C G Am F C Am F G Am Bbadd9 C G Am F G C Am .

F F Dm Bm F C G Dm F C G Dm F C G Dm F C G Fsus2 G Dm F G C Am F C G Dm F C G F C G Am F C Gsus4 C Fsus2 Fsus2 Fadd9 G Fsus2 C Em Am7 G Fadd9 C Am7 G Fadd9 C Bbadd9 F G Am F C G Dm F C G Am F C G Dm F C G Am F C Gsus4 G Am F C Gsus4 G Fadd9 F Am G C Am F C G Am F C G F C Am G F C G F C G Am F C G Dm F C G F C G Am F G C Am F C G Dm F C G Dm F C G Am F C F C G Am7 D C C G Fadd9 F C Fadd9 Am G C Dm7 F C G Dm F C G Am F G C C G Fadd9 Dm7 F C G Dm F C G Am F G C F Am G F Am G F G Gsus4 C G Am F C G Dm F C G Am F C F Dm F F C F C G Am F C G Am F Am G F C C Em Fsus2 C Gsus4 Am7 F C F Am C G C G Am C C Am7 Fadd9 C Am7 G Fadd9 G Dm F Bb Gsus4 C C G Fadd9 Fadd9 G F C G F G G Dm F G C G Dm F C D Am F G C Am G Bbadd9 Am F G Am F C G Dm F C G G Am F C G Am F C G Am F C Am G F C G Dm F C G F C G F F Am F G C Dm Csus4 C G C F Am F Am Gm C G Am Csus2 F C G F C G F C G Am F C Am F C G Dm F C G Am F G C G Dm C Csus2 F C G Am7 Fadd9 C Am7 Fadd9 F C G Am7 Fadd9 C G Am7 Gm C G F C G F C G Dm Csus2 F C C F Am F F C G Dm F C G Bm F F C G Am F C G Em Am7 G Fadd9 C C Am7 G Fadd9 C G D Fadd9 G Fsus2 Am C F C Am F G C F F G Am F C G Am7 G G F G C Dm F G C C Gsus4 Fadd9 Fadd9 Am7 G Fadd9 Gsus4 G F G Bm F Am G F G G C Dm F G Dm F C G5 C Fadd9 Am7 Fadd9 Fadd9 G Fadd9 G F Am G F C G Am F C G F G A Am7 C G C Am7 Fadd9 C Am7 G Fadd9 C G Fsus2 C G Fsus2 Gsus4 G C Am7 Am7 Gsus4 C Gsus4 Gsus4 C G Dm F C G Dm7 F Fadd9 G F G Fadd9 G Am F C F C G Bm Am F C G Dm F G C G Dm F C G F Am F C G Am F G C G F C G Dm F C G Am G F G G Dm F F C G .
C Dm7 C Gsus4 Fadd9 G Am F C G Dm F G C Dm7 C G C C G Am7 G Fadd9 C Am7 Fadd9 C Fadd9 G C Am F C Am F G Am F G C G Dm F G C Am F C G C C G Fadd9 F C G G5 G C C Fadd9 Gsus4 Fadd9 C G Am F C G D Am F C G F Am G F Am G F C G Dm C Em G F Am7 G Am7 G Fadd9 F F C G Am G F C G Am F F C G Dm C F Am G F Am G F C G Am F G C Dm F G Am F G Am G F G C G Am F C G C G Am7 A G C Am G F Am G Bm F F Am F Am F C F Am G C Am Csus2 C Gsus4 C C C G Fadd9 C G Am7 G G Dm F Am G Am F C G Am F C G Am F C G Am G F G Gsus4 G C G Dm F F C Dm F C G Dm F F C G Am F G Am A G C Am G5 G Fsus2 Am7 Bb G Am Bbadd9 C Am F C G Dm F Am F C G Dm F G G Dm F G G Dm F C G Dm F G C G Dm F C Am G F G D Am G Dm F C C Em Fadd9 G Bbadd9 Fadd9 G Am F G Am F C G Dm F C G Am F G Am F C Am G F C G C C G7sus4 C F C G F C G Dm F C G Dm C F C Fadd9 G Am7 C C Am7 Fadd9 C G Fadd9 G Fadd9 C G C Am7 G Fadd9 G Fadd9 C G Am7 Fadd9 C Am7 G Fadd9 Gm Am C F C G F C G Am F C G F C Em Am7 Fadd9 C G Am7 Fadd9 C G F G Gm Am F C G Dm F C G Dm F C G Am F C G Dm F G G G F Am G F F Bm F C G Am F C Am G F C G Am F C C F Fadd9 G Am F C G Am7 Fadd9 C C Am7 G Fadd9 C Am Fadd9 C Am7 Fadd9 C C Am7 Fsus2 Fadd9 G Fsus2 C Dm F F G F C G Am F C G Dm F C G Dm F C G Am G F C G Dm F C G F C G Am F C Am C C Dm7 G F G G Dm F G C Dm F C G Am F C A C Fadd9 Am7 G Fadd9 C G Fsus2 G A G Am F C G C G F C G Am F G Am C G Csus2 G F Bbadd9 Gm F G C C Am C Bbadd9 Em Am7 G Fadd9 C Am7 Fadd9 Gsus4 Fadd9 G G Fadd9 G G Am F C C Am7 G Fadd9 C G Fadd9 G Fadd9 G5 C Gsus4 C G F G C Am7 Fadd9 Fadd9 C Fadd9 Fadd9 Fadd9 Fadd9 C Fadd9 C Em Am7 G Fadd9 C Bb Gsus4 C Gsus4 Fadd9 C G Am F C G Am F G C Am F F C G F C G Am G F Am F G C G Dm F C G Am F C G Dm F C G F Am G F Am G F Am G C F F C G Am G F C Am G F Dm Csus2 F G G Dm7 F C G Am7 .
F G Am F C G Am F C G Dm F C G Am F F G C G Am F C Fadd9 F C Fadd9 Am F C G Dm F C G Dm F G C Dm F A F G Bbadd9 F G C Dm F G C G Am F C G Am F C G F C G Dm F C G Dm F C G Am G F C C Am7 G G F C G Dm F G7sus4 F C G Dm F C G Dm F Am G F F C G Am F C Am F C Am G Am7 C G C G Dm F C G Fsus2 C Am7 Fadd9 G C Am7 Fadd9 C Gsus4 G G Dm F C G F C G Am F F G Csus4 F F C G A C G Dm F G5 G G7sus4 C F C G Dm F D C G Dm F G C Am G F F Gsus4 Fadd9 Csus2 F C G Dm F C C G C Am7 Am7 Fadd9 G C C Am7 G Fadd9 C G Am7 G Fsus2 C G F G C G Dm F C G Dm F Bb G F Am G F G C A Am7 G Fsus2 Bm Dm7 F C G Dm F G C C G Fadd9 G Dm F C G Am F G Am G F Bb G C Am7 G Fsus2 G C Am C Gsus4 G Fadd9 C F Am C F C Gsus4 C Fadd9 C Gsus4 G Fadd9 F G C G Am F C F G Fsus2 Am G C C Fadd9 Fadd9 Fadd9 G Gm C C F C C F Fadd9 G Dm F C G Am G F C Am G C F C F F C G F G F Am G F C G Dm F C G G Dm F C G Fsus2 C G Fadd9 Fadd9 Csus4 F Am F G G Dm Am F Em C Am7 G Fadd9 C G Fadd9 Am7 Fadd9 G Am F C Am F C C G7sus4 F G C G Am F C G F C G Dm F C G Dm F C G Dm F C G Am F C C Dm F G C C G C Am7 G Fadd9 C G F C Am G F C G F G Am F C C Am F C G F C G A F G Am F C G C C Em Fadd9 G Am F C G Am F C G Dm F C G Am F C G Am F C G Am7 F G C C Am7 Fadd9 Fadd9 Fsus2 C Fadd9 G F C G Dm F G C Dm F G G Am F C G Dm F C G A Am F F G C C G Am7 Fadd9 Fadd9 Gsus4 C C Am7 F C Gsus4 G C Am7 G G F Am G Am G F C G Am F C G F C G Am F Dm F C G G F G C G Dm7 F G C C Am7 G Fadd9 C Am7 G Am G Am F C C G Fadd9 G Dm F C G Dm F C G Dm F C G Dm F Am G F C C C Fadd9 G Fadd9 C Am7 Fadd9 Fadd9 Fadd9 C Gsus4 G G F Am F C G G5 G C C Am7 Fadd9 C Gsus4 C C C Am7 Gsus4 Fadd9 C G Fadd9 G Am C G C Gsus4 Em Fadd9 G Fadd9 G Am7 Am G C F Am G F Am G F C G Dm F C G Dm F Em G Am F C G Dm F G Am F C G F C Em Dm F C G Am F C G Em G Am F C G Dm F Am G F Am G F Am G Dm F G C Gsus4 Am F C C Am7 G5 G C C G Am7 Gm C C F Fadd9 G F C G Dm F C G Am F C Fadd9 G Fadd9 C Am7 C Gsus4 Fadd9 G Dm F C Gsus4 Dm Csus2 F C G Dm Csus4 F C G Am G F C G Am C F Dm Gm Am F C G F C G Dm F F G C G Am F C G F Am G F C G Dm C F F C Am G F C G Dm F G Am F G Am7 G Am F G Gsus4 F C G Am F C G F C G Am F C G Dm F C G Am G C C Am7 G Fsus2 G C Am7 G Fsus2 C Am Bm F C G C Am F C G Am F C C G Am7 Fadd9 C G Fadd9 Am7 C Am7 G Am F C G G Fadd9 G Am F G C G Am F C G Dm F C G Dm F C G Am F C G C Fadd9 G Dm F C G Dm F Dm F C G Dm F C G Am F C G Dm F G C G Dm F C G G Dm F F Am G F C G Am F C G Am7 Fadd9 C Gsus4 Fadd9 G F Am G Am F G Am G F C G C Fadd9 Fadd9 Gsus4 C C Am7 Fadd9 G C .
Em Am F C C Am7 Fadd9 Fadd9 Am7 G C F C G Dm F C G Dm F Am G F C G F C G Dm F C G Am F C G Am F C Am F C G C Em Am7 Fadd9 C G Am7 Fadd9 C G Fadd9 G Am F G C G Am7 G Fadd9 C G Fadd9 Am7 Fadd9 C G Am7 Fadd9 F Am G Dm F G C C Am7 Am7 G C C Am7 G Fadd9 C Fadd9 Am7 G C C C Am7 Fadd9 Gsus4 C G F Am G F C G .
C F Fadd9 G Fsus2 C G Am F C F Dm C C Am7 Fadd9 C Am7 Am7 C C Am7 Fadd9 C C Am7 G Fadd9 G Am F C G5 G Fadd9 C Gm C G Dm F G C Dm7 F C G F Am G C G F C G C G Am F C C G Am7 Fadd9 C G F Am F C G Dm F Dm C F Am F C G Am F F G C Dm Csus2 F C Am7 Am F C G F C Em Am7 Fadd9 C Gsus4 Bbadd9 C G G Dm F C G Am C C G Am F Em C G Fsus2 Am7 G Am F Am G Dm F G G F C G Am F C G Dm F G G Am F C G Dm F Am F C G Am F C G Am F G C Dm F G F F C G Dm F G F F Am Gsus4 F G C Am F G Am Csus2 G C C Am7 G Fadd9 G Am7 F C G Am G F C G Am F C G Dm F G Am F C G Am F C Am F F C G Dm F C G Dm F C G Dm F C G Dm F C G F G C Am F C G F C Am7 G Fadd9 C F C G F C G Am G F G Bm F C C C Am7 .

Fadd9 D Fadd9 G Am F C G Dm F C G Am F C G C Fadd9 F C G Dm F C F Fadd9 G C Am7 G Fadd9 C G Fadd9 G Am7 G F G F D G Dm F G C Dm F G Am F C G Dm F C C G Dm C G7sus4 F C G Dm F C Bb G C C G Am7 G Fadd9 C Am7 G G C C G G7sus4 F C G Dm F C G Am F G Am F C Gsus4 Gsus4 C Am G F G Am F G Am G F F Am G F Am G C Bbadd9 F G C G Dm F G C G Fsus2 Am G F G C Am F G Am F C G Dm F C G Dm F C G F C G Am7 G G Am F G C F Fadd9 Gsus4 G C G F Am G F C G Am F C .
C C Fadd9 F Fadd9 C Am7 G Fadd9 C Gsus4 G C G Dm F C G Dm F G Bm F C G F C G Dm F C G Dm F G G Am F C G Dm F C G Am F C G Am F C Am F G Am F C G Dm F C G Fsus2 C G Dm Csus2 Csus2 F C Am F G Am G F F G Dm C F G C G Am F C C Dm F G C G Dm F G C G Am7 G .
C Fadd9 Fadd9 C Am7 Fadd9 C G F C G Dm F C F C Fadd9 G Fsus2 Bbadd9 F G Dm F C G Dm F G C G Dm F C Am F C G Dm Am F C G Dm F C G Dm F C C Dm Am G Am F G Am F Csus4 C C Am7 Fadd9 G Am Am F Am G F Am G F Am G F G7sus4 G F Am F G C Bm C C Am7 G F C G Am F G Am F C Am F C G G A C G Am C Em C Am Am7 C G Am7 Fadd9 C Am7 Fadd9 C Fadd9 G Fadd9 C Am7 Fadd9 G F Am Csus2 F C G Am Bbadd9 C G Dm F C G Am F C G Dm F C G Dm F C G Am F C G Dm F C G Am7 G Am F G F G C G Fsus2 G Dm F C G Dm F G C G F C Em A C G Dm C C Am Gsus4 C Am F G C Am F G Am F G C Dm F C G Dm F C G Dm F C G Am F C Am G F Am G F Csus2 G Dm Am F C G Am7 G Gsus4 C G Fadd9 G Fadd9 C G F C G Am F C G Am F C Am F F Am G C F Am G Fsus2 Am G F C G Dm F G Bm F Am G Am F C G Am F C G F F C G Dm F C G Am Am7 Gsus4 F G Am G F C G Fadd9 F Am G F Am G G7sus4 F C G Am Fadd9 F C G F C G Dm F G Bm F C G Dm F C G D A C G Am F C G F G C G G Am F G F G Bb F C C Csus2 F C Am7 G Fadd9 Fsus2 Am7 G Am C F C Am7 G F C G Dm F C G Am F G F Am G F G Am Csus2 F C C G C G5 Fadd9 C G C G Am F G C G Dm F C G F C G Dm .

G Bb A G C Am7 G C Bbadd9 F Am G F Am F C G Am F C G F C G Am F Bm C F C G F C C Gsus4 C G Am F C D F C G Dm F C G Dm F C G Dm F C G F C G Dm F C G Am F G Dm F G C G Am F C G Am F G C F C Gsus4 G F C G Dm F C F Am F G D Am F C G Dm F G C Dm F G G Am G F Am G F Am D F Am7 G Gsus4 C G Am C G Am F C G C F G G F C G F C G Am F C C F C Gsus4 Am C F Am F C G Am F C F C Bm F G C G Dm A F Am G C F Am F C G Am F G G Am F C G Dm Fadd9 F C G Dm F C G Am F C G Dm F C G Am F G C G Dm F C G F C Bb G C F G G F Am G F C G F C G Am F C F Dm F C G Am F C G Dm F C G Dm F C G Dm F C Am F C G F C G F C G Dm F C G F C G Am F C G Am F G Dm7 F F G F C G F C G Am F C G Am F G Am F C G F C G Am C F Am F F G Bbadd9 F Bm F C G Dm F Dm F C G Am F C Am F G C G Dm F C Fsus2 C Fadd9 Fsus2 C Am F Am D F C G Dm F G C Dm F C C F Am7 G C Am C G C G5 Fadd9 C C Fadd9 .

C Fadd9 F C G Dm F C G F C G Dm F F G Dm F C G Am F C G Am Bm F C G Am F C Gsus4 Am F G C Dm Am F G Am F C G Dm F G G Dm F C G Dm F C G Dm F C G A C G Am F C G A Dm F G G Dm F G C G Dm F Dm F G C Dm F C G C Am7 G C C Am7 G Fadd9 Fadd9 Am7 G Fadd9 C Fadd9 Am7 G Fadd9 C G C C C Am7 Fadd9 G F Fadd9 G Am F C C Fadd9 G Fadd9 Fadd9 C G Fsus2 G Dm F C G Dm F C G Dm F C G C C Am7 Fadd9 C G Fadd9 G Am7 C Em Am7 Fadd9 C Am7 Fadd9 C G Am7 Fadd9 G7sus4 Dm7 F Fadd9 G C Am F C G G Fsus2 G G Fsus2 G C G F G D F C G F Am G F Am F C G G7sus4 F G G Am F C Bm F G Am F C Am F G Am F C G Am F G C G Em Gsus4 C Gsus4 C Gsus4 C G Dm F G C Am C G Am F Dm F C G Dm F F Am G F G G F C G Dm G5 G C C Dm Fadd9 C C Fadd9 C G Fadd9 Fadd9 G Am F G Am F F C G Am7 G Am7 G G C Csus4 F C G Dm F C G Dm F C Am F C G Csus4 F F C C Am7 G Fsus2 C G Am F F C G Bm G F C G Am G F Am G F C Dm F G C G Am F F C G Am F C G Gm Am G C C G Dm F G D D Bbadd9 F G C G Dm F C G Fadd9 G Fadd9 G G7sus4 F F C G Am F C G Fadd9 C Am F Fadd9 G F C G A Dm F C G Am7 G Fsus2 C G Am F C G C Am7 C G C Am7 Fadd9 Fadd9 G F Fadd9 C G Am7 Am C Gsus4 Csus4 Gsus4 F C G Am F G C C Dm F C F C G Am F C G Dm F C G F G C G Am F G C C Em Am7 Fadd9 Fadd9 C Fadd9 Am7 F C G Am F G C G Dm F C D F C G F G Am F C G C D C Am7 C C Am7 G Fsus2 Am G F C G Dm F G C Dm F C G Dm F G Am F C Dm Dm F C G Dm F G C Am F G C G Am F C G F G G C G Dm F G C C Dm F G Dm F C G Dm F Am G C C Am7 Fadd9 C G Am7 G G Dm F F C G C F D Fadd9 G Dm F C G C Dm F C G Dm F C G C Fsus2 Fadd9 Gsus4 G C Dm F G C Fsus2 Fadd9 F G Fsus2 Am G F Am Em G Am F Bm C G Dm F C G F G C C Am C Gsus4 G G Am F Dm F C C C Am7 G Gsus4 C G Am F C G Am F G C Am F C G C Fadd9 Am7 G G Am7 G Gsus4 C Fadd9 G Fsus2 C G C Gsus4 Am F G Am F C G Dm Csus2 F C G F C Em D C G Dm F Am Gm G C Dm F G C F Am G C Dm F C D F C G F F C Am Gm C C F C G Am F C G C C Dm Fadd9 C F Fadd9 G Am F C G Am7 Am7 G C C G Fadd9 Bbadd9 G Fsus2 C Fsus2 C Fadd9 Am7 Fadd9 C C Am7 Fadd9 G C Am7 G Fadd9 C G Am7 Fadd9 C C G5 Fadd9 Fadd9 Bbadd9 Fadd9 C Am7 Fadd9 C G Am7 G Am7 Bm F C G C G Dm F C Am F G Am G C G Am F C Bm C G Fadd9 F C Gsus4 Dm F Am F Am F C Am7 Gsus4 Fadd9 G F C G Am F C Am F C G Am F G Am C G Gm Am G F Bb G C C Fadd9 G C Am7 G Fadd9 C Am7 Fadd9 Fadd9 Am7 Fadd9 D C G F C G F C Am F Am F C G Am F C G Fadd9 Am7 Fadd9 G Fadd9 C Am7 Fadd9 Gsus4 C C G Fadd9 G Fadd9 Fadd9 C Am7 Fadd9 C C Fadd9 Am7 C Gsus4 G G Dm F C G C Am7 Fadd9 Am7 G Fadd9 C G Fsus2 G Am F C G G5 Fadd9 C Am7 G Fadd9 C G Am F C G Dm F G C Dm F G C G Dm F C C Em Dm7 G F C G Am F G C F C G Am Gsus4 F G C Am F Csus2 F C G F C G Dm F C G Am F G Am F C Dm7 F C G Am F C Am F C G C Am F F Am F G C Gsus4 G Fadd9 G C Am7 Dm7 C Fadd9 Am7 C C Am7 Fadd9 Gsus4 Fadd9 G Am7 G Fsus2 Am7 C Gsus4 C G C Dm F Am7 G Am F C G F C G Am F G F Em G Dm F C G F C G Am F C G C C G Am F G C Am G Fadd9 A C Fadd9 Fadd9 G Fadd9 C C Am7 F C G F C G Dm F C G F Csus2 G C Dm F G C Am F Fadd9 Csus4 F C G C Fadd9 G Fadd9 C Am7 G Bbadd9 C G Am F F C G Dm F C G Am F Am G F C G Dm F G Am F C C Am7 Fadd9 C Am7 G Fadd9 C Am7 G Fadd9 F Bm F Am G C Dm F C G Am F G Am G F C G F G C Dm F G C .

F Fadd9 G Dm F G Dm F C G F Am G F G Csus4 F G G Dm Dm Csus4 F Am G F C G Am F F G Am F C C G Fadd9 C Am7 D Fadd9 C Fadd9 G Fadd9 C Fadd9 Fsus2 Fadd9 G F C G Am F C F Am G F C G Am F C G Am F C G Dm F C C Fadd9 Fadd9 C C G Am7 Fadd9 G Am7 Am F C G G G Dm F C G C C Fadd9 C Fadd9 Am7 F C G Am F C G Dm F C G Dm F C G F Am G F G C Dm7 F C G Dm F Csus4 G C C Am7 G Fadd9 C Bbadd9 Fadd9 G F C G Dm F C Fadd9 G Fadd9 G Dm F C Dm Am F C Am7 G Fadd9 C Fsus2 Am7 F C G Dm F Am G F F C G C Fsus2 C C Am7 Fadd9 G C Fadd9 G F C G Am F G C Dm F F G Am F C G C Am7 Fadd9 C Fadd9 C Am7 Fadd9 C G Am7 G Fadd9 C G Fadd9 G Fsus2 C G C Am7 G Am7 Am C F C Fadd9 F Fadd9 C G G7sus4 F F G C G Am G F Am F C G Dm F C G Dm F C G Am F G F G G7sus4 Csus2 F Am F C G Dm F C G Am F C G G F Bm D Dm F C G Am7 Fadd9 G Am7 G Fadd9 G Dm F C C G Fsus2 Am G F Am G C F G F F C G Dm7 F C G Fsus2 G Am C C G5 Gsus4 Fadd9 G Fadd9 G Am F C Gsus4 C G Am F C G Am7 Fadd9 Fadd9 G Am7 Fadd9 C Am7 Am G F Am G F F Am G C Am F C G Dm F G C G Dm F G C G Dm F C G F C G Am7 Fadd9 Gsus4 F Fadd9 G F C G Am C F C G Dm F C G Dm7 F G Am F C Am Csus2 G C Am7 G C F Am F C G Dm F C Dm G7sus4 F G C G Dm F G C Dm F G C Am F C Am F G C C G Am7 Am7 C G Am7 Fadd9 C Fadd9 G Am7 F C Gsus4 G C Am F C G Dm C G Dm F C G Dm F C G Dm F C G Dm F C G C Fadd9 Csus4 C C C Am G F D Dm C G Am F G C G G F C Am F G C G G Am F C G A C G Am F G .

F Fadd9 G Am F G C G Am F C G Am F C G F Am G5 G C G Am F Am G F Am G F F G F Dm F F G G Am F C Am G Fsus2 F Fadd9 G Am F C G Dm F Am F G Am F C G Dm F G C F C G Am F C Am G F G Fadd9 G Am F G C G Am F G C G F G C Em G Gm C C Am7 Fadd9 Am7 G C Am7 G Fadd9 G Am F C G Dm F C G C D Am F G G Dm A G Dm F G C Csus4 Csus2 F F Am G C Am G C Am F G C G Am7 F C C Fadd9 G Am F C Am F F Am F C G Dm F C G Am F G Am F C G Am F C G D C G Fsus2 C G Dm C C Gsus4 C G Dm F C G Dm C F C Gsus4 G Am F C C Am7 Fadd9 Gsus4 C C .
C Am7 Fadd9 C C Fadd9 G Fadd9 Gm Am G F Am G F Am G F C G Am F Am7 G C Am7 G Gsus4 Am G F C G Dm F C G G Am C G C G Am F C G Fadd9 C G Dm F G G Dm F C C F C G Am F C G Am F C G Dm F C G Dm F C Gm F Am Bb G Dm F C G Am F Am G F C G Am F C G Dm F C G Fsus2 G C C G Fadd9 G Dm F C G Am F C F C G Am C C Em Gsus4 Fadd9 G Am F Am F C G Am F C G F C F C Am7 C G Am7 G7sus4 F C G Dm F G C G Am F G G Am7 G G5 G F F C G F Am G Am F C G Am F G Am F F Am G F C G G Dm F G C Dm F F C G F C G F C G Dm F G C Am F C A F F C C G Am F C G F C G Am F C C Fsus2 Fadd9 G Am7 F F Am C G Am F C G C F Am G C Am F G Bm Dm F F G F Am G C Dm7 Bm F Dm F G C Dm F C G F C G C G Dm F G C G Dm F C G Dm F G C F Am G F G C Dm G Dm C Gsus4 F C Am G C F G C G Am F C G G7sus4 F G C Em Am G F G C F Am Gsus4 C G Am F C G Am G F Am Bb C C Fsus2 Fadd9 G C Em Am7 G Fadd9 C Fadd9 G .

Am7 G Fadd9 C C Am7 Fadd9 C G Fadd9 C Em G Am7 G Dm7 G Dm F C G F C G Am7 Fadd9 C Gsus4 Fadd9 C Gsus4 Fadd9 G Am F G Am Bb G C F Am F C C Gsus4 Fadd9 G Bbadd9 C G G F Am G F C G Am F C G Dm Am F C G Dm F G C Dm F C G Am F F C G Em Dm F G C F Dm F G C Csus2 F C G Dm F C G F C G F Dm7 G G Dm Am C C Gsus4 Fadd9 Fadd9 G Fsus2 C G Dm F C G Am F C G F C G F C G Am F G C G Gm Am F F G Am F G D C Dm F C G Dm F G G Fsus2 G Dm F Em C Am F Am F C C G Am F C G Dm C F Fadd9 Bm F C G Am F C C G Am7 G Fsus2 C Em G Fsus2 G G F Dm G F Am Dm7 F C F Am G F G G Gm Am G C C G Bm F C G Dm F C G Am F C G Am F C G Dm F G C G Am F C G C Dm C Em Am F G C G Bbadd9 Dm C F G C G Am G F Am F C F C G Fsus2 Dm7 F G Am F C C G Fadd9 Csus4 G5 G G Dm7 F Am Gsus4 F C G F Am F C F Am G F G F Am G F Am G F F Am G F Fadd9 G C Gsus4 Fadd9 G Fsus2 F G A Am7 G Am C G Bb G D C G Dm F C G Dm F C G Dm F C G Dm F C G C Gsus4 Am7 G C Dm F G C G F C G Am F G C G G5 F C C G Fadd9 Fadd9 C C Am7 G G Bm Am G F G F Am G F C G Am F G C G Dm C F Am G F Am G C G Dm F C G C Gsus4 G C Am F C G Am F C G F G C Dm F G C Dm C G Am F C G Dm F C Am G F Csus2 G F Am G F Am G F C G F C G F C G Dm C F Am G Bb F C Fsus2 C Fadd9 Am7 G Fadd9 C G Fsus2 C G Am F G C G Am C G C Dm Csus2 F G C Dm F G C F Am G F C G F C G Dm F C G Dm F C G Am F C G Em Bbadd9 Am G C Am F G Csus4 F C Gsus4 Am G F Am G C F Fadd9 G F C G Am F C G Dm F C G F G C Am F G C G Dm F C G Am7 G C C Dm Fadd9 F G C G Dm F C G Csus4 F C Gm Em Am .

G Fadd9 G Am7 Em Fadd9 Am7 Fadd9 G G7sus4 F C G Am7 C G Fadd9 F C G Am7 Fadd9 C Fsus2 C Am7 Fadd9 C C C Am7 G Fadd9 G Dm F G C G Dm F G Bm F G C Dm F G C G Fsus2 G Am7 G Am A F Am G F G C G Am F C G C Dm F G Am F G C G Dm F G Am F C G Am F C G Dm F Dm F C C F Am G F C G Am G F Am G F G C F C G Dm F G Dm F C G Am F G C Am F C G Dm F Dm C F C G Dm F C F C Fadd9 G Am7 G Am C G Am F C Dm7 F C Gsus4 G Fsus2 G Csus4 F Fadd9 G Am C C Fadd9 C C Am7 Fadd9 Fadd9 G Am7 G Fadd9 C Am7 Am7 C G C D F C G Dm F C G F Am F C Dm F G Am F C G Am Bb G F C G Am F G Am F C Am F C G Dm F C G Dm F C G Am F C G F C G C .
C Am7 C Fadd9 G C G Am F C G Dm C G G Am F C G Dm F C G Am G F Am G F C G Dm F G C C G Am F C G Am F C G Dm F F C G A C G Dm F C G Am Gm C G Dm F C G Dm G5 G Am F C C G Fadd9 G F C G F Am G C F C Dm F G C G Dm F C G Am F C G Am F G C Csus2 C G C Am7 G Fadd9 C G Am F C .

F C G C F G Dm F C G Am7 F Am C F Am G F G Am7 G Dm7 C Dm Am F C C A Fadd9 G Am F C G7sus4 F C G F Dm F C G Dm Dm7 F C G Am F Am F G G5 Fadd9 G Fsus2 G Am G G F Dm F C G Dm F C G Am7 G Fadd9 C Am7 Fadd9 C Am7 Fadd9 Gsus4 C G F C G Am F Dm F G Am Csus2 Bm F C G Am F C G Am F G C Am F G C G Dm F C G Am F C G Csus2 F C G Dm F C G C Am7 C G Fsus2 G7sus4 Fadd9 F Am G F C G Dm F C G F C G Dm F G C G Dm F C G Dm F G F C G Am F G C G G5 Fadd9 C Am7 Fadd9 C G Fadd9 .
Fsus2 C G Fadd9 G F C G Am Fadd9 C Fsus2 Fadd9 Em G Am7 G G Am F F Am Em F G Bm F G C G Am F C G C C Fadd9 Fadd9 Fadd9 G Am7 Fadd9 Fadd9 G Fadd9 Am7 C Gsus4 Am7 G C Am7 G Fadd9 F Bb G G Dm F C G Dm F C Am F C G Dm F G C Dm F C G F Am G C F Fsus2 C G Dm F C G Am F C G Dm F C G Dm F C G Am F C G Am F G Am F C G Am F C G Dm F C C G F C G Dm F G C G Dm F Dm F C G Am F C G F C C G Fsus2 G F C G F Am G C C G Fadd9 G G C Fsus2 G Fadd9 G Fadd9 Gsus4 C G Am7 G Am F Am G F Am F C G Dm7 F F C C Gsus4 Fadd9 Fadd9 G C Am7 G A Am G F C G Am F C C Dm C C G Am7 G G C Em G Fadd9 F C G Dm F G C G Dm F C C F Fadd9 Fadd9 C G Am7 C G C Gsus4 F Csus4 C C Am7 G Fsus2 C C G Am7 Fadd9 C G Am7 G Fadd9 Dm7 F C G F C G Dm F C G F C G Am F F G C G Dm F C G Am A F C C Gsus4 Fadd9 Fadd9 G Fadd9 F Bm Fsus2 D Dm F C Dm F G Dm F C G Dm C G C Gsus4 Fadd9 G Fadd9 C G Am7 C G Am F C G Am F Fadd9 G Am F Am F F G C G Bm C G Dm7 G Dm C F Am G F Am G F Am G G F Am G F Am F Am C G Dm F C G Am F C Am G C C Fadd9 Fadd9 Gsus4 G C Am F G C G C C G Fsus2 G C Am7 Am7 C G Am7 G Fadd9 C G Dm F C G Dm F C G Am F Fadd9 G F C Fadd9 C G Am7 G Fadd9 C G Fsus2 C C Fsus2 Am F C C Bm Dm7 G Dm F C Am .

# Ed Sheeran
Am Db C Am F C G Am C Dm F Am Bm C F F C Am7 Am Fadd9 C Am G C G F C G D7sus4 Am C Am7 Dm Dm F Am G7sus4 C C G Am Gsus2 F#m7 Am G7 Gsus2 Am Fsus2 Fsus2 Gsus2 F#m7 Em Fsus2 F# G Fadd9 Dm7 Dm F G C Dm F F G C C G Am Fadd9 C Gadd9 Fadd9 Bb Am Dm F Fadd9 Am F G C Fm Fadd9 C G F Bm Bm D7sus4 C Am Db C C5 C Am G7 Am Dm7 G7 Am Fm C C G Am F Bb C C G Fadd9 Am Em C G F Am F C Gadd9 G Am7 Am Dm F G Am Dm F G C Dm Am F G C C Fsus2 Am Dm F G Am Db C F Em Em F G C D F C Am7 F C Am7 Am F G C Gsus2 F Am Em C G F G Am C G G7 .
F# F C F G C Dm F G Fsus2 C G Em Am Bb C Am C5 Am Fm C G C Am7 F G C F G C C Am G C G C5 Am F#m C Am A C Dm F G Em Am Am7 G D7sus4 C C G Gsus2 Am Em Fsus2 Gsus2 C Em F Fadd9 Dm7 C F G Am C F F C Am7 C Am F C Am F C G Am Fadd9 C G Fsus2 D7sus4 C C A Am E C G Am Gadd4 C G Am G F C G F Am F C Dm7 Am F G Am C Fsus2 .
Am Em Fsus2 Am Am C G F G C C G Am F# G Dm7 Dm .
Am C7 Fm C C G Am Fadd9 C C5 C Am C5 D7sus4 Gadd9 Db C Am F C Gsus2 F .
Am Em Fsus2 Am Fadd9 C G F Dm Dm F G Am C G7 F F C Am7 F#m7 Am C C G Am Em C5 Em F F F C G Am Fadd9 F C C G F C Em F G Am C G F G C F G C F# G F G C C5 F Am Fadd9 C G7 Fsus2 C D Gadd4 C G Am Fadd9 F C Gadd9 G Fadd9 Am Em A F C C F F C G F# Fadd9 C C F Am C Am F F G C Dm F G C Fm Gsus2 Am C .
Am Am Fadd9 G7sus4 C D C G F Fm C C G Am Fsus2 F G C C G Am Fadd9 C G F Fadd9 C5 C C F Am C A Fm Am Fm C Am F C Dm F Gsus2 Am Am Fsus2 Am Am F G C C G Am Gadd9 G F Dm Dm F G Am C F G Am C F G G7sus4 C Am G C D7sus4 F C Fsus2 Am G7sus4 C Gadd4 F G Am Db C Dm7 F G C Am7 F Am C G F C C F Am Em Fsus2 Am Am F G Am Dm F G C C Am F D7sus4 E Dm F G Am A D C Am7 F Am7 F#m C C7 F# G F G C C F Am G7sus4 C Am F C Am Fadd9 C G F Am Em C G Gsus2 Am A C Fm C Am Fadd9 C Am F C Dm7 Am F C Am F G Am E C F G C C G Am F G C Dm F Bb Am F#m7 C E G F# G F G C G Fadd9 Gadd9 A Gsus2 Am Em C F G C C G Am Fadd9 C Am Fadd9 C Am F G G7sus4 C G F#m F# G C Am F C Dm F F C C Am G A C Am F C G Em Am G Am E G Am F G C C F Am G C G Em Am Gadd9 G F C G F Gadd4 F#m7 C G F Am C Gadd9 G F# G Fsus2 C G Am G Fm C F Fadd9 C Fsus2 F#m Dm Fadd9 C Am C C G F C G C Am G Fm C G F G C Am Em G Am Fadd9 C Am7 Am Fadd9 C Am C5 C Dm F G Am Dm F Dm F C G F# G C Am F C Am F C C F G Bb C G7sus4 G Am C G Am Gadd9 C5 Dm Dm F G Am C Dm F C Am F G A C F G7 C Dm Fadd9 Am Em Fsus2 F G D Gadd9 G Dm7 Am D7sus4 C E Fsus2 C G Db C G Em Am G C Dm F Fsus2 Am C7 Fsus2 G7 Dm F G C C G F C Am7 F Am C C Am F G Am C G F C G Dm7 G Am Dm7 F G Am F#m7 C G Am Fadd9 Fsus2 C Am7 Am Am7 Fsus2 Bm Am7 Am7 Gadd4 F C Am7 F Am C Gadd4 Gsus2 Am Am C G F C G F G Db C Dm7 Fadd9 Am Dm F Fm Am C G F Am Dm7 C G Am Em C G Fadd9 Am Dm F G Am Dm F G Am C G F G C G E Fadd9 C Bm F Db C C G Am .
Am Fadd9 C G Am Fadd9 C Am F C Am Am7 C Am Fadd9 C Am F Db C C7 Am7 Am Em C7 F C G D7sus4 C Em F G C C F Am C Gadd9 G F Am C Fsus2 Am Am G C C F F G C C G Gadd4 G F#m7 C C Am G C G F Dm C5 C E Fadd9 C Am F C Db C Dm F Fsus2 Am C Dm F C G Bm .
A C C5 Em Dm7 F C Am G Am C G F C G .
F# Fadd9 C C F G C7 F# F C Am F C Am Am F G Am D C Am F G Am Fm C C Gadd9 G F C F# G F C G F Bm F#m7 C G Am F C Am F Fadd9 Am Am F Am C F G C C F C5 Am Dm F G Am Dm C G F# C5 C Am Em F G F#m C Am7 F C Am7 F Am C C G Am Am F G C F Em Am G Fsus2 C Em F C Am7 Am Dm7 Fsus2 F Am C G Am Fadd9 C Am F C G Am Dm7 C5 C Am C7 G F# G C G Am Fadd9 C Gadd9 G C Gsus2 F G C G Am F#m C G Am G C G F C G Em Gadd4 F C Am F G C .
Am C G7sus4 Em Am7 Dm7 Bm F#m7 Am7 C Am F C G Gsus2 Am Em Fadd9 Am Fadd9 C Dm G7sus4 Fadd9 C C G Am Dm F G Am Dm F G Am Fadd9 C C F G7sus4 C C Dm7 Am F# G7 F G C F#m Am F C Dm7 F C F#m Am C C5 Am Am G C G Fadd9 C G C7 F C Dm7 Am F G Am F D C Am F C Am Gadd4 F G G7 Gsus2 Am Gsus2 C Am A C G Bm Fsus2 F# Dm D7sus4 C Am F C F C G Am F G Am Am7 C F G7sus4 C Am7 Db Am Em Gadd4 F G Am C5 F Em Fsus2 Am Dm Gadd4 G G D C C7 F Am Dm7 C A Am Gsus2 C G F C G F Am G C G Am Am F Dm Am F G C C F G C G7 F G C C G Am G Gadd9 Am G Fm A F#m7 Fm C .
Am F#m7 C Am F C Am7 F Fadd9 C C G Fsus2 Am Em G7 Am Fadd9 Fsus2 Dm Em G F C C Bb Am G C G F G7sus4 C C Am Dm F G Am D Dm7 Am .
Am G Fsus2 Am Em Fsus2 Fm Am Em F G Am F C C Dm Fadd9 Fadd9 C Fm F#m Am Am F G Am Dm F G C G7sus4 Am C G Am G7sus4 C C Am Am G F#m7 G C Dm7 F C G Am Fadd9 C Dm F C Bm Am F C Am F G Am C F G Am C G7 F Fadd9 C C Am7 Am Fadd9 C Am G Gadd9 G C G F G C C G G7sus4 Am C Am7 Am F C Am Fadd9 D C C7 C5 Am F C Am F C Am F C Am Fm C G Am Dm F G Am Dm G7 F C Gadd9 F G C F Am G7sus4 C G Dm7 Fadd9 C G Am D C Am C7 A C G F Am G C Am G F# G C F F Fsus2 Am G7 C G F#m7 .
Em Am7 D7sus4 Am C G F C C Am F C G F Am C C5 Am Am F# G7 Am Am A C Am F C Am Gadd9 G Am Dm F# G F Dm .
Am Em G F G C5 C Fsus2 Am Db C C F Am .
Am F#m7 Gadd4 C5 C Am F C Am Dm7 Fadd9 C Dm F G C Am7 .
Gsus2 Gsus2 Am Am F G Am Dm F F C Gadd9 G Am Dm F F G C G F C C Gadd4 Am G Am Fadd9 C C G Am E F G C C G Am Am F G Gsus2 C7 C5 Fsus2 A C E G Am Dm Fsus2 C D7sus4 G Am C F G C Am F C G F C G Am Em G F C G F C G F C Fsus2 G7 Am Dm F G Am G7sus4 C Gadd9 G F C C Am F Fadd9 C C F# G Fadd9 C F#m7 C Gadd4 G7 G7 C7 Db C C7 Fm D Am Em Fsus2 F Db C Dm F G Am C G F C C7 F C G F C Am7 E D7sus4 C C5 .
Am F# G Am C Bb F Am C G F Am Gadd4 Fadd9 C G Am Dm D F .
C Gadd9 G C Dm Fadd9 Am Fm C G7 F Gadd4 A C Am F C D7sus4 Am G C Am G F C C F G C C F Dm7 Am G C Fm Gadd9 C G Gsus2 Am Gsus2 Fsus2 Am Fsus2 C G D7sus4 Am C C7 F C G E D C Am F C Am F C G Am E F G Gadd4 C7 Em Dm F G Am G C Am G Fm C G F# Dm F G Am Bb C G Gsus2 Am Em C G Am C G Am Fadd9 F# Gsus2 F# Dm F G Am C F G F C Am7 Am G C G Am Fadd9 C Am G F C G7sus4 Am Db C F#m7 G Am Dm F G Am C G Gsus2 Am Em Fsus2 F Bb C G Fsus2 Am F Fsus2 Am F G Am Bb C G F C C G Am Fadd9 C Dm F G Am Em Am G F Bb C Dm F C G Am G7 C F#m7 F C F#m7 Am C Fsus2 Gsus2 Am Fadd9 .
Am Em Fsus2 Am Bm G Am G7 F G Fsus2 C Bb F C C G Am G F Dm C G F Dm Fsus2 F#m G C Am7 Bm F C A Am F G C Am7 F A C Dm F Gadd9 G C Dm F G Am C5 C G F C C7 Gsus2 C Am7 C7 A C Dm F C G F C G7sus4 Am Gadd4 G F C7 Am G7 F G C C F Am G7sus4 C G F C G Am Am7 Fsus2 Am Fadd9 Fsus2 C C7 E Am F C Am F G Am C F F C C E G Bm C Fsus2 F#m7 Fadd9 Fsus2 .
Am Gadd4 Fsus2 C G F F#m7 Am G7 F G D7sus4 Am Am C A Dm7 Am F G Am Am Fsus2 C Am F G7sus4 F# Fadd9 C Am Dm C G F G C G Gsus2 C F Fadd9 Am C G Am Dm F G Am Bb C Am F C G C Am F C Am F C D Am Fadd9 C C G Am Am F G C G7 F C5 C7 C G F C Bm C Db F#m C F G C C Bm Am G C G Am Fadd9 C Am G F# G F C G F#m Am C G F C F#m7 G C F# G C Gsus2 Am Db C Am G C G F Am C G Gadd4 E G7sus4 Am Dm C G Em Am F C G7sus4 Am C G Am Db C D F# F Fsus2 Am Em Fsus2 Am Gadd4 G C G F G7 C C F G Am Am7 C Em F Am C G F C Gsus2 F C Fsus2 C Dm F Fsus2 Am F#m7 F#m C C G Am G Gadd4 G7sus4 C C G Am D F G C C7 F C D7sus4 Dm7 C G Am F C Am F G C G Am Fadd9 C C F Dm7 G7sus4 C C Am7 Am F C Dm F C G C F Gsus2 Am Am Fsus2 C Am F C Am F C Am F G C7 G7 F G C C F Bb C F G C Dm F C Bm F Am C G F C G F G C C F Am C C G Am G7sus4 C C F G Am F#m7 C F Bm Dm F G Am Fsus2 C G Bm Am .

# Adele
Em Emadd6 G Am F C F Am F G Am F G F G Esus4 F G Am Am G Am Bb F G Em F G#m Dm Cm A5 Csus4 Ab C7 Gbm Csus4 Am C G Am F G F F Am7 C Gsus4 Esus4 Gbm Eaug G C Dm7 G Am A C4 F7b5 E7 Am F C C Fm Gsus4 Dm7 A5 Am G#m G Em7 Cm Gsus4 Esus4 F9 G Am Bsus Csus4 A C Am C G Esus4 F7 Dmadd9 G Dmadd9 G Am Cm F E Am F Dm C G D Am Em Gbmadd6 D C F C F F Am F Am F G F G F F G Am G G C F G F G Am Dmadd9 C G F Am G F Am G Bb C E5 Am F B Csus4 F7b5 G E7 Bb7 G C G G Bbbm F G B Am G C7 Am D7sus4 G9 G Am F C Am F F9 F Am7 Am Em G Em Dm G Am F F Asus4 G Am7 Emadd6 G Am G G7 D G Am G7 G Am F G F G F F C Em Am F G Am F G Ab F Am F Ebm Bsus F A Bb7 E7 Am F G Am D7 G F Esus4 G Em D Ab G Am F Ab7 F C Csus4 F C Em7 C F C C G Am E5 Eaug F C F Am Am F F G Am F7 F F D7sus4 D C Asus4 C E7 Am F G Am7 F G Am C Ab7sus4 Am Dm7 G F7 F G F G Am E5 Am G G Am G C G F Dmadd9 G Am Gadd9 G Eaug F7b5 C Ab7 F Am F C F G Em Am G F F F Dm Am G5 Am Gbmadd6 G Am D Csus4 C D C7 G7 G F7b5 F G Am F G G F C G Am G D7sus4 G Am7 G Gbm7 Em C A5 Ab G B E7 Gsus4 Ab7 Ab7 B Am F F Bb7 C A5 Emadd6 G Am F E F C Em Eaug Bb F Am Am Emadd6 C Fm Ab7sus4 Am Cm C F C G Am F G Bb G G4 Am F G Em G B Am Bsus Dm Am C C G Am F G G G Dm Bsus Am A7 G Am7 G .
C F Am G Emadd6 D7 A7 G Am F C G Am F C G C Cm E G4 F Esus4 G E5 Dm7 G G Dm G G Dmadd9 Gadd9 Em F7 F B Am G Am7 Am G Am F Ebm G Am Em F Gsus4 G9 Ab C F Am G E7 Em Dm7 A5 C A5 Am Am Eaug F G G Am G G Am Gbm Am F G Am F G#m C G F G F Csus4 C C Dmadd9 Em7 Bsus .
G A7 F C F Ab G Am7 Am C C G G7 F G G Ab7sus4 Am Em Em C G9 D7sus4 G G7 Ab G F F C G Am F Ab7sus4 G7 Csus4 G Am G#m G Em E G4 C G Am G F C E Am G F F Am7 G7 Emadd6 C F A F7b5 G G#m F G#m Am F G F F C F A5 Am B Csus4 C4 C Dm7 B F9 Esus4 C4 Am D7sus4 C Am G C F G Ab7sus4 Emadd6 F Asus4 Bb7 Eaug G4 G Am F G9 G Am F Csus4 .
G Gbmadd6 G C G G Dm Am F G F C C G G G#m Am C Dm C D7sus4 C G Am Am E Am F G F Ebm F Ebm Emadd6 G Am Am F Bb C Gbm7 A7 F C Em Am A7 Dmadd9 G F7 Am G Am G F C G A G Bbbm G Am F Fm F F7b5 Gbm7 Am Gbm Am C G F G F G F G G F G F F E7 Am F Gbm G#m F G F G Am G7 F C G Am F Ab7 F Dm D Am G5 C C G Am F Esus4 F G#m F C Am Ab7sus4 Ebm Em D7 Emadd6 G Am C G C .
Gbm7 G Am Am G Am Em G4 G#m F9 Am Am G F G F Bb G Ab G D7sus4 G Am F G F G Am F G F Gbm Am F G G Am G F G Bsus F C C E7 B F Em7 Em7 A Gsus4 Ebm A E Am Em G Ab7sus4 Am G C G C7 Am F G Am Ab7sus4 Am A7 G C C Am F F F Am E5 Am G G5 Am7 G G4 G Am G C4 C Cm F Ab7sus4 F G7 Gbm Am F G7 F Em Am Bb7 G Am Am F F G G#m G G7 Bbbm Am F7 F G Am G F F9 G Gsus4 Am D C G F C F Ab7 F7b5 Ab7sus4 Am D7 Am G Emadd6 F C Am C F G Am Em G Am G Em E G Am A5 Am Cm Esus4 G C C F G G4 C G5 Am G E Am F C Dm7 Bb7 .
G C Eaug Gadd9 Ab G G4 F G Am D7sus4 C Gadd9 F C Am Am Am G F G Asus4 G Am F Am F G Em Em7 G G9 G G D7sus4 Gbm7 C G F7 F C C G7 Am F C Dm G F C G Am G G F Bsus F G Gadd9 G G Em G E5 Gbmadd6 G F7b5 Am G F C C F E7 Bb C F7 F7 G G Am A F F E Am G F G G7 G G C G C Em G Am F G G#m F Em Ab7sus4 Ebm A5 C4 Ebm D A7 Bb C Am7 Am Em C F C Am C7 D7 Eaug G9 G F F G Am F C Gbm7 Em F Dm Ab7 G Am G Am G G F E F G D C G Am Am G Am G Eaug F F F C4 D F G Dmadd9 G Cm A5 Gbm7 Emadd6 F7 Ab7 Am G Am F G9 G C G Am Fm C G Am C C G Am F Ebm C Am G7 G F G Gsus4 G F G D7sus4 G Am C4 G Em7 Dm7 Gbmadd6 G Am G F9 G Am7 Gbmadd6 Esus4 F F G4 Dm7 Am Em Am G Am F F F F C Gadd9 C Am Am F C G Am F F G Csus4 G9 G Am F7 G F G Gsus4 F G Am Em Am Em G7 G7 C G Am F G Am A G Em .
Am C Am Am Dmadd9 G Ab G G Dmadd9 G A Am Em G Eaug .
Em7 Gadd9 F Gsus4 G G Am7 F G5 Gbmadd6 C C G Eaug F G Am Am C F G G Cm Dm Gadd9 G Am F G Gadd9 G G G G D F C F Cm Gbm Em Csus4 Gsus4 Am A .
F Asus4 G Am F C Dmadd9 C F C F C7 F Csus4 C Emadd6 G Dm C7 Csus4 C C Gbmadd6 F Am F7b5 F G Am D C Em Am F Em Dm7 F C C Gadd9 C Am Csus4 G G5 Cm F7b5 Em A7 G Am7 G Ab A7 C E7 Am Am G Am G D Csus4 F Am Emadd6 C Dm Eaug G Am Am Am Bsus F E7 E .
C Em Am F C Dmadd9 C Am B Am G7 G Am G G G Em A5 A5 Em Bbbm Gsus4 Fm Fm E7 Csus4 Em G Em F Ab G Am G A G G F G Am G F G5 C F Am Am G Am F C F9 G F G Am F C C E Am F C G Am F G G9 F A5 C G Am F C C Am F G Asus4 G Am G Dm7 G B Am Em F F A5 Dm7 C G Am C G F C G Am Bb7 G F Cm G Am G4 C C Em D G Am A Dmadd9 Am Fm G D7 E5 Bbbm Bb7 Dmadd9 G4 Bsus Am G C G Gbmadd6 C G Am Am7 F F Dm F G Am G C F C G F C C G F G Dm Gbm7 F7 Dm C F C Ebm E5 F C G Am Cm G A C Eaug Bb F Fm Em Em F7b5 Em7 Cm G A7 A Am Dm Am7 F C Csus4 D7 Am F C Am G A7 G Bb G F Am C F C D7 Am G4 G Am F C Am7 F F Am Am G4 Ab7sus4 C4 D7sus4 Bb Fm F9 Gsus4 C F Ab C C E Am F C D7sus4 G Esus4 G G Csus4 G Am F Em Ab Dm Ab Em Gsus4 C Dm C C7 Am Am G C Ab7 Dm G Ab7sus4 G G Gbm7 A5 F G F9 G Dm7 F Dm7 G Am C Dm Am Em Dm C Ab C F B Am Em A5 Dm D7sus4 C7 C7 F F C C G Gbm7 C G C G Am7 F Am F Am Am G F C F C G Am Em C G Em Bbbm F C G F F G Em Am F G Bsus Em G A5 E7 Dm G#m C7 Am G F C G F G C7 G7 F G Am G .
C G C F G D G Bb7 A5 Am Bb Gbm7 F Am G Am F G Am F C D7sus4 Esus4 G9 Am F F Ab7sus4 Am G C Em G#m Esus4 G F Em B G7 Ebm C Am G F G Em7 G Emadd6 .
C D7 Am Bbbm C G Am F G Am G Am F C G Am Gsus4 C G G F F C G Bb F E Am F Am F Em7 C A5 Am Gsus4 Gbmadd6 G Ab G C F D F Dm G Gbm7 F Ab7 Em7 C F F9 G Am Am G G#m E Gbmadd6 Gsus4 Ab D7sus4 Am7 G Ab7 G Am F C C C Am F F Ab7 G Am F C G F F Am7 F7 C7 Am Dm7 G C G Gsus4 Em7 Bb B Am F G9 Ab7 Gbm7 F D7 G7 G C G Am F C F F Bbbm A Am Ab7 G Dm G D7sus4 Gbmadd6 G Am F G Am F C G F7 F G D Am G5 Cm Em Am Gbmadd6 C Gbm Am7 F Ab7 Bb7 F C Em C Eaug C G G7 G D D7 Gbmadd6 Csus4 G4 Dm Am Am C G Ab7 G G G D7 F G Am G Gbm7 C C C G Am F G Am F C G B F A5 G9 G Am Am F C C G C G A F G Cm G Gbm7 Ab7 G Am F Dm7 C G F F Dm7 C Am F C G D G Ab7sus4 F9 A A C4 Ab7sus4 Asus4 D7 G A5 Gbm Ab7 F G Am C G Bsus Am G Ab7 Gbm Am G Bb7 F Dm7 G Am7 G Am Gbmadd6 E Asus4 G G#m G F G F G G5 G Am F C G F G F Asus4 .
G Am F G F A5 C G C7 D7 G Dm C Emadd6 Am Am G A5 F G Am Em C G F7 G Dmadd9 F Dm7 G Am C C C4 Gbmadd6 Em7 Gbmadd6 Am7 Am Am G Esus4 G Am F .
Am F C Am C A5 C G D Ab7sus4 Bbbm C7 Am F F C7 D7sus4 C C C C C G F D7 F G C G Am G F G D7sus4 Eaug C C G Am G F G Gbm7 C G G E F C Dm C Asus4 C C G Am F C Ab7 F Am C C C Csus4 Dm7 Cm Am G G F G F F G Am C Am7 F B Emadd6 G A F7b5 G C Em Gbm Em7 Csus4 E5 Gsus4 Eaug Ab C4 Am Gbm Am F C C G Am G G Am E F F .
C G Am G F F C F Ab C D C F C Em Am Asus4 Emadd6 G Am G5 G F F Ebm A G4 E5 G5 Dmadd9 Gbmadd6 G5 Am Em C Em F7b5 C Ab7 G G F G Em G9 G A5 G G5 Dmadd9 D7 Cm Am E5 Am G G A C G Bbbm A5 Am F G A5 Em F Gsus4 Bbbm Bb7 C .
Em F Dm7 B Am G F7b5 G Am F Emadd6 G F Dmadd9 G G7 Am E Cm G G9 Eaug G Am F C G Am F G Am Em C Am E Am G G G Am F G Am C A5 Am Am Bsus Am F F D7 Am G Bbbm Asus4 D Am Eaug Ebm C F Am F F C C C G Am F G F F .
Am F C G Am C C G Am F G Am F C G Gbmadd6 C F C F Gadd9 G C Ab E Am G F G D7sus4 F Asus4 .
E7 Am F F Am G C G C7 Bb G A7 G F C G F Csus4 Fm Bb7 Bb Gbm F7 F F C G D F Em7 Ab F G F7 Cm C A Esus4 F7b5 Gbm Bsus C Em7 C G Am7 E Em Am G Am F D Ab7sus4 G4 .
Ebm G#m G Emadd6 C F Am G#m C F F Fm C Gbmadd6 C Am F G G#m Am F C F F G C G F F D7 Am G F G .
C Gbm7 Am F .
